<a href="https://colab.research.google.com/github/AndikaPutra509/Prediksi-Saham/blob/main/IDX_Scanner_Manual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# =============================================================================
# IDX STOCK SCANNER - V3.4 (PHASE 3 - FINAL FIX)
# Dengan: ADX Fix | Filter Modal Kecil | Google Sheets URL Tetap Muncul
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import warnings
from datetime import datetime, timedelta
import time
import math
import pickle
import os
from tabulate import tabulate
from collections import defaultdict
import logging
import random
import gc
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive, files, auth
import hashlib
import json
import requests
from scipy import stats
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import seaborn as sns
import gspread
from google.auth import default

# Matikan semua logging yang tidak perlu
logging.getLogger('yfinance').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore')

print("="*60)
print("🏦 IDX STOCK SCANNER - V3.4 (PHASE 3 - FINAL FIX)")
print("   ADX Fix | Filter Modal Kecil | Google Sheets URL Tetap")
print("="*60)

# =============================================================================
# 1. KONFIGURASI TETAP
# =============================================================================

# Stock Universe (sama persis dengan asli)
STOCKBIT_UNIVERSE = [
    "AALI", "ABBA", "ABDA", "ABMM", "ACES", "ADES", "ADHI", "ADMF", "ADMG", "ADRO",
    "AGAR", "AGII", "AGRO", "AHAP", "AIMS", "AISA", "AKRA", "AKSI", "ALDO", "ALKA",
    "ALMI", "AMAG", "AMFG", "AMMN", "AMRT", "ANDI", "ANJT", "ANTM", "APEX", "APIC",
    "APLN", "ARCI", "ARGO", "ARII", "ARNA", "ARTA", "ARTO", "ASBI", "ASDM", "ASGR",
    "ASHA", "ASII", "ASLI", "ASMI", "ASPI", "ASRI", "ASRM", "ASSA", "ATLA", "AUTO",
    "AVIA", "AWAN", "AYLS", "BABP", "BACA", "BALI", "BANK", "BAPA", "BAPI", "BATA",
    "BAYU", "BBCA", "BBHI", "BBKP", "BBLD", "BBNI", "BBRI", "BBRM", "BBSS", "BBTN",
    "BBYB", "BCAP", "BCIC", "BDMN", "BEEF", "BEKS", "BELL", "BEST", "BFIN", "BGTG",
    "BHAT", "BIMA", "BINA", "BIPP", "BIPI", "BIRD", "BISI", "BJBR", "BJTM", "BKSL",
    "BLTA", "BLUE", "BMAS", "BMBL", "BMRI", "BMSR", "BMTR", "BNBA", "BNBR", "BNGA",
    "BNII", "BNLI", "BOBA", "BOLT", "BOSS", "BPFI", "BPII", "BPTR", "BRAM", "BREN",
    "BRIS", "BRMS", "BRNA", "BRPT", "BSDE", "BSIM", "BSSR", "BTEL", "BTON", "BTPN",
    "BTPS", "BUDI", "BULL", "BUMI", "BUVA", "BWPT", "BYAN", "CAMP", "CANI", "CARS",
    "CASA", "CASS", "CBDK", "CBMF", "CCSI", "CDAX", "CEKA", "CENT", "CFIN", "CITA",
    "CITY", "CKRA", "CLEO", "CLPI", "CMNP", "CMPP", "CMRY", "CNKO", "CNTX", "COAL",
    "COCO", "COWL", "CPIN", "CPRO", "CSAP", "CSIS", "CTBN", "CTRA", "CTTH", "CUAN",
    "DAAZ", "DART", "DASA", "DAYA", "DCII", "DEGA", "DEWA", "DFAM", "DGIK", "DIGI",
    "DILD", "DIVA", "DIVN", "DKFT", "DLTA", "DMAS", "DMND", "DNAR", "DNET", "DNLS",
    "DOID", "DOOH", "DPNS", "DPUM", "DSFI", "DSNG", "DSSA", "DUCK", "DUTI", "DVLA",
    "DYAN", "EASI", "EASY", "EBMT", "ECII", "EDGE", "EKAD", "ELBA", "ELSA", "ELTY",
    "EMBR", "EMDE", "EMTK", "ENRG", "ENVY", "ENZO", "EPAC", "EPMT", "ERAA", "ERTX",
    "ESSA", "ESTA", "ESTI", "ETWA", "EXCL", "FAST", "FASW", "FILM", "FISH", "FITT",
    "FKSF", "FLMC", "FMII", "FORE", "FORU", "FORZ", "FPNI", "FREN", "FUJI", "FUTR",
    "GAMA", "GDST", "GDYR", "GEMS", "GGRM", "GGRP", "GHON", "GIDS", "GJTL", "GLVA",
    "GMFI", "GMTD", "GOLD", "GOOD", "GOTO", "GPRA", "GRPH", "GSMF", "GTBO", "GTRA",
    "GTSI", "GULA", "GZCO", "HADE", "HDFA", "HDIT", "HEAL", "HERO", "HITS", "HKMU",
    "HMSP", "HOKI", "HOMI", "HOPE", "HOTL", "HRME", "HRTA", "HRUM", "HSBK", "HSMP",
    "HUMI", "IBFN", "IBOS", "IBST", "ICBP", "ICON", "IDPR", "IFII", "IFSH", "IGAR",
    "IIKP", "IKAI", "IKAN", "IMAS", "IMJS", "IMPC", "INAF", "INAI", "INCF", "INCI",
    "INCO", "INDF", "INDS", "INDX", "INDY", "INET", "INKP", "INPC", "INPP", "INPS",
    "INRU", "INTA", "INTD", "INTP", "IPCC", "IPCM", "IPOL", "IRRA", "ISAT", "ISEA",
    "ISSP", "ITIC", "ITMG", "JAST", "JAWA", "JAYA", "JECC", "JEMB", "JFAS", "JGLE",
    "JHON", "JIHD", "JKON", "JKSW", "JMAS", "JPFA", "JPII", "JPUR", "JRPT", "JSKY",
    "JSMR", "JSPT", "JTNB", "KAEF", "KAQI", "KARW", "KBLI", "KBLM", "KBRT", "KBRI",
    "KDSI", "KDTN", "KEEN", "KETR", "KICI", "KIJA", "KINO", "KIOS", "KJEN", "KKGI",
    "KLBF", "KMTR", "KOBX", "KOIN", "KOLI", "KONI", "KOTA", "KPAL", "KPIG", "KRAS",
    "KREN", "KRYA", "KSEL", "KUAS", "KUIC", "KUVO", "LAND", "LAPD", "LATA", "LBAK",
    "LCGP", "LCKM", "LEAD", "LIFE", "LINK", "LION", "LISA", "LMAS", "LMPI", "LMSH",
    "LPCK", "LPGI", "LPIN", "LPKR", "LPLI", "LPPF", "LPPS", "LSIP", "LSPI", "LTLS",
    "LUCY", "MABA", "MABH", "MAGP", "MAIN", "MAMI", "MAPA", "MAPB", "MAPI", "MARA",
    "MASA", "MAYA", "MBAP", "MBCA", "MBMA", "MBSS", "MBTO", "MCAS", "MCPI", "MCOR",
    "MDIA", "MDKA", "MDKI", "MEDC", "MEDS", "MEGA", "MERK", "META", "MFIN", "MFMI",
    "MGLV", "MGNA", "MGRO", "MIDI", "MIKA", "MINA", "MIRA", "MITI", "MITT", "MKNT",
    "MKPI", "MLBI", "MLIA", "MLPL", "MLPT", "MLSL", "MMIX", "MMLP", "MNCN", "MOLI",
    "MPOW", "MPPA", "MPRO", "MPTJ", "MRAT", "MSIE", "MSIN", "MSKY", "MTDL", "MTFN",
    "MTLA", "MTPS", "MTSM", "MUDA", "MUTU", "MYOH", "MYOR", "MYRX", "MYSX", "NAGA",
    "NASI", "NATO", "NAYZ", "NCKL", "NELY", "NETV", "NFCX", "NICL", "NIKL", "NISP",
    "NITY", "NIYM", "NOBU", "NPGF", "NRCA", "NSSS", "NTBK", "NUSA", "NUSI", "OASA",
    "OCTN", "OKAS", "OMED", "ONIX", "OPMS", "ORNA", "OTBK", "PADA", "PADI", "PAMG",
    "PANR", "PANS", "PANU", "PAPA", "PASA", "PASS", "PBRX", "PBID", "PBSA", "PCAR",
    "PDES", "PDGD", "PDIN", "PEGE", "PGAS", "PGLI", "PGUN", "PICO", "PIDRA", "PJAA",
    "PKPK", "PLAN", "PLAS", "PLIN", "PMJS", "PMMP", "PNBN", "PNBS", "PNIN", "PNLF",
    "PNSE", "POLI", "POLL", "POLU", "POLY", "POOL", "PORT", "POWR", "PPGL", "PPRE",
    "PPRO", "PPSI", "PRAS", "PRDA", "PRIM", "PRIN", "PRLD", "PROD", "PROT", "PRTS",
    "PSAB", "PSBA", "PSDN", "PSGO", "PSKT", "PSSI", "PTBA", "PTDU", "PTIS", "PTMP",
    "PTPP", "PTPW", "PTRO", "PTSN", "PTSP", "PUDP", "PURA", "PURE", "PWON", "PYFA",
    "RACE", "RADIO", "RAFI", "RAJA", "RAKD", "RALS", "RANC", "RATU", "RBMS", "RDTX",
    "REAL", "RELI", "RIGS", "RIMO", "RISE", "RMBA", "RMKE", "ROCK", "RODA", "ROKI",
    "ROTI", "RRMI", "RUIS", "RUMI", "SABA", "SAFE", "SAME", "SAPX", "SARA", "SATO",
    "SBAT", "SBBP", "SBGA", "SBMA", "SBMF", "SCBD", "SCCC", "SCCO", "SCMA", "SCNP",
    "SDPC", "SDRA", "SEAN", "SECR", "SEMA", "SFAN", "SGER", "SGRO", "SHID", "SHIP",
    "SIDO", "SILO", "SIMA", "SIMP", "SIPD", "SIPO", "SKBM", "SKLT", "SKRN", "SLIS",
    "SMAR", "SMDR", "SMGR", "SMIL", "SMMT", "SMSM", "SMRA", "SNLK", "SNMS", "SOFA",
    "SONA", "SOSS", "SOUL", "SPMA", "SPMI", "SPNA", "SPRE", "SPTO", "SQBI", "SQMI",
    "SRAJ", "SRIL", "SRSN", "SSIA", "SSMS", "SSTM", "STAR", "STTP", "SUGI", "SULI",
    "SUPR", "SURI", "SWAT", "SWID", "TALD", "TAMA", "TAMU", "TAPG", "TARA", "TASP",
    "TATA", "TAXI", "TBIG", "TBLA", "TCID", "TDPM", "TELE", "TEMB", "TEMPO", "TIFA",
    "TIGA", "TINS", "TIRA", "TIRT", "TITA", "TKGA", "TKIM", "TLKM", "TMAS", "TMPO",
    "TMSH", "TOBA", "TOOL", "TOPS", "TOSK", "TOTL", "TOTO", "TOWR", "TPIA", "TPMA",
    "TRAM", "TRGU", "TRIO", "TRIS", "TRJA", "TRON", "TRST", "TRUB", "TRUK", "TRUS",
    "TSPC", "TUGU", "TURI", "TUVN", "TYRE", "UANG", "UCID", "UDIJ", "UFNX", "UGRO",
    "UJSN", "ULTJ", "UNIC", "UNIQ", "UNIT", "UNSP", "UNTR", "UNVR", "USFI", "VALU",
    "VICO", "VICI", "VIDI", "VISI", "VIVA", "VKTR", "VOKS", "VRNA", "VTNY", "WAPO",
    "WEGE", "WEHA", "WICO", "WIFI", "WIIM", "WINS", "WMUU", "WMPP", "WOOD", "WOWS",
    "WRKR", "WSBP", "WSKT", "WTON", "YELO", "YULE", "ZBRA", "ZINC", "ZONE"
]

# Quality Stocks (sama persis dengan asli)
QUALITY_STOCKS = [
    'ADRO', 'AMRT', 'ANTM', 'ASII', 'BBCA', 'BBNI', 'BBRI', 'BMRI', 'BREN', 'BUMI',
    'CPIN', 'EMTK', 'GOTO', 'ICBP', 'INCO', 'INDF', 'INKP', 'ISAT', 'JPFA', 'JSMR',
    'KLBF', 'MDKA', 'MEDC', 'MIKA', 'MTEL', 'NCKL', 'PGAS', 'TLKM', 'TOWR', 'UNTR',
    'AKRA', 'ARTO', 'BBTN', 'BDKR', 'BIRD', 'BJBR', 'BJTM', 'BRIS', 'BRPT', 'BUKA',
    'EXCL', 'HRUM', 'INTP', 'ITMG', 'MPMX', 'PTBA', 'SIDO', 'SMGR', 'TPIA', 'UNVR',
    'WIKA', 'ACES', 'ADHI', 'AGRO', 'AALI', 'ARNA', 'ASGR', 'ASRI', 'AUTO', 'BAYU',
    'BEST', 'BFIN', 'BIPP', 'BISI', 'BKSL', 'BLTA', 'BMAS', 'BMSR', 'BNGA', 'BNII',
    'BOBA', 'BOLT', 'BOSS', 'BPFI', 'BRAM', 'BSDE', 'BSSR', 'BTON', 'BUDI', 'BULL',
    'BUVA', 'BWPT', 'BYAN', 'CAMP', 'CANI', 'CARS', 'CASA', 'CASS', 'CBDK', 'CBMF',
    'CCSI', 'CDAX', 'CEKA', 'CENT', 'CFIN', 'CITA', 'CITY', 'CKRA', 'CLEO', 'CLPI',
    'CMNP', 'CMPP', 'CMRY', 'CNKO', 'COAL', 'COCO', 'COWL', 'CPRO', 'CSAP', 'CSIS',
    'CTBN', 'CTRA', 'CTTH', 'CUAN', 'DART', 'DASA', 'DCII', 'DEGA', 'DEWA', 'DGIK',
    'DIGI', 'DILD', 'DIVA', 'DIVN', 'DLTA', 'DMAS', 'DMND', 'DNAR', 'DNET', 'DNLS',
    'DOID', 'DOOH', 'DPNS', 'DSFI', 'DSNG', 'DSSA', 'DUCK', 'DUTI', 'DVLA', 'DYAN',
    'EASI', 'EASY', 'EBMT', 'ECII', 'EDGE', 'EKAD', 'ELBA', 'ELSA', 'ELTY', 'EMBR',
    'EMDE', 'ENRG', 'ENVY', 'ENZO', 'EPAC', 'EPMT', 'ERAA', 'ESSA', 'ESTA', 'ESTI',
    'ETWA', 'FAST', 'FASW', 'FILM', 'FISH', 'FITT', 'FKSF', 'FLMC', 'FMII', 'FORE',
    'FORU', 'FORZ', 'FPNI', 'FREN', 'FUJI', 'FUTR', 'GAMA', 'GDST', 'GDYR', 'GEMS',
    'GGRM', 'GGRP', 'GHON', 'GIDS', 'GJTL', 'GLVA', 'GMFI', 'GMTD', 'GOLD', 'GOOD',
    'GPRA', 'GRPH', 'GSMF', 'GTBO', 'GTRA', 'GTSI', 'GULA', 'HADE', 'HDFA', 'HDIT',
    'HEAL', 'HERO', 'HITS', 'HKMU', 'HMSP', 'HOKI', 'HOMI', 'HOPE', 'HOTL', 'HRME',
    'HRTA', 'HSBK', 'HSMP', 'HUMI', 'IBFN', 'IBOS', 'IBST', 'ICON', 'IDPR', 'IFII',
    'IFSH', 'IGAR', 'IIKP', 'IKAI', 'IKAN', 'IMAS', 'IMJS', 'IMPC', 'INAF', 'INAI',
    'INCF', 'INCI', 'INDS', 'INDX', 'INDY', 'INET', 'INPC', 'INPP', 'INPS', 'INRU',
    'INTA', 'INTD', 'IPCC', 'IPCM', 'IPOL', 'IRRA', 'ISEA', 'ISSP', 'ITIC', 'JAST',
    'JAWA', 'JAYA', 'JECC', 'JEMB', 'JFAS', 'JGLE', 'JHON', 'JIHD', 'JKON', 'JKSW',
    'JMAS', 'JPII', 'JPUR', 'JRPT', 'JSKY', 'JSPT', 'JTNB', 'KAEF', 'KAQI', 'KARW',
    'KBLI', 'KBLM', 'KBRT', 'KBRI', 'KDSI', 'KDTN', 'KEEN', 'KETR', 'KICI', 'KIJA',
    'KINO', 'KIOS', 'KJEN', 'KKGI', 'KMTR', 'KOBX', 'KOIN', 'KOLI', 'KONI', 'KOTA',
    'KPAL', 'KPIG', 'KRAS', 'KREN', 'KRYA', 'KSEL', 'KUAS', 'KUIC', 'KUVO', 'LAND',
    'LAPD', 'LATA', 'LBAK', 'LCGP', 'LCKM', 'LEAD', 'LIFE', 'LINK', 'LION', 'LISA',
    'LMAS', 'LMPI', 'LMSH', 'LPCK', 'LPGI', 'LPIN', 'LPKR', 'LPLI', 'LPPF', 'LPPS',
    'LSIP', 'LSPI', 'LTLS', 'LUCY', 'MABA', 'MABH', 'MAGP', 'MAIN', 'MAMI', 'MAPA',
    'MAPB', 'MAPI', 'MARA', 'MASA', 'MAYA', 'MBAP', 'MBCA', 'MBMA', 'MBSS', 'MBTO',
    'MCAS', 'MCPI', 'MCOR', 'MDIA', 'MDKI', 'MEGA', 'MERK', 'META', 'MFIN', 'MFMI',
    'MGLV', 'MGNA', 'MGRO', 'MIDI', 'MINA', 'MIRA', 'MITI', 'MITT', 'MKNT', 'MKPI',
    'MLBI', 'MLIA', 'MLPL', 'MLPT', 'MLSL', 'MMIX', 'MMLP', 'MNCN', 'MOLI', 'MPOW',
    'MPPA', 'MPRO', 'MPTJ', 'MRAT', 'MSIE', 'MSIN', 'MSKY', 'MTDL', 'MTFN', 'MTLA',
    'MTPS', 'MTSM', 'MUDA', 'MUTU', 'MYOH', 'MYOR', 'MYRX', 'MYSX', 'NAGA', 'NASI',
    'NATO', 'NAYZ', 'NELY', 'NETV', 'NFCX', 'NICL', 'NIKL', 'NISP', 'NITY', 'NIYM',
    'NOBU', 'NPGF', 'NRCA', 'NSSS', 'NTBK', 'NUSA', 'NUSI', 'OASA', 'OCTN', 'OKAS',
    'OMED', 'ONIX', 'OPMS', 'ORNA', 'OTBK', 'PADA', 'PADI', 'PAMG', 'PANR', 'PANS',
    'PANU', 'PAPA', 'PASA', 'PASS', 'PBRX', 'PBID', 'PBSA', 'PCAR', 'PDES', 'PDGD',
    'PDIN', 'PEGE', 'PGAS', 'PGLI', 'PGUN', 'PICO', 'PIDRA', 'PJAA', 'PKPK', 'PLAN',
    'PLAS', 'PLIN', 'PMJS', 'PMMP', 'PNBN', 'PNBS', 'PNIN', 'PNLF', 'PNSE', 'POLI',
    'POLL', 'POLU', 'POLY', 'POOL', 'PORT', 'POWR', 'PPGL', 'PPRE', 'PPRO', 'PPSI',
    'PRAS', 'PRDA', 'PRIM', 'PRIN', 'PRLD', 'PROD', 'PROT', 'PRTS', 'PSAB', 'PSBA',
    'PSDN', 'PSGO', 'PSKT', 'PSSI', 'PTDU', 'PTIS', 'PTMP', 'PTPP', 'PTPW', 'PTRO',
    'PTSN', 'PTSP', 'PUDP', 'PURA', 'PURE', 'PWON', 'PYFA', 'RACE', 'RADIO', 'RAFI',
    'RAJA', 'RAKD', 'RALS', 'RANC', 'RATU', 'RBMS', 'RDTX', 'REAL', 'RELI', 'RIGS',
    'RIMO', 'RISE', 'RMBA', 'RMKE', 'ROCK', 'RODA', 'ROKI', 'ROTI', 'RRMI', 'RUIS',
    'RUMI', 'SABA', 'SAFE', 'SAME', 'SAPX', 'SARA', 'SATO', 'SBAT', 'SBBP', 'SBGA',
    'SBMA', 'SBMF', 'SCBD', 'SCCC', 'SCCO', 'SCMA', 'SCNP', 'SDPC', 'SDRA', 'SEAN',
    'SECR', 'SEMA', 'SFAN', 'SGER', 'SGRO', 'SHID', 'SHIP', 'SILO', 'SIMA', 'SIMP',
    'SIPD', 'SIPO', 'SKBM', 'SKLT', 'SKRN', 'SLIS', 'SMAR', 'SMDR', 'SMIL', 'SMMT',
    'SMSM', 'SMRA', 'SNLK', 'SNMS', 'SOFA', 'SONA', 'SOSS', 'SOUL', 'SPMA', 'SPMI',
    'SPNA', 'SPRE', 'SPTO', 'SQBI', 'SQMI', 'SRAJ', 'SRIL', 'SRSN', 'SSIA', 'SSMS',
    'SSTM', 'STAR', 'STTP', 'SUGI', 'SULI', 'SUPR', 'SURI', 'SWAT', 'SWID', 'TALD',
    'TAMA', 'TAMU', 'TAPG', 'TARA', 'TASP', 'TATA', 'TAXI', 'TBIG', 'TBLA', 'TCID',
    'TDPM', 'TELE', 'TEMB', 'TEMPO', 'TIFA', 'TIGA', 'TINS', 'TIRA', 'TIRT', 'TITA',
    'TKGA', 'TKIM', 'TMAS', 'TMPO', 'TMSH', 'TOBA', 'TOOL', 'TOPS', 'TOSK', 'TOTL',
    'TOTO', 'TPMA', 'TRAM', 'TRGU', 'TRIO', 'TRIS', 'TRJA', 'TRON', 'TRST', 'TRUB',
    'TRUK', 'TRUS', 'TSPC', 'TUGU', 'TURI', 'TUVN', 'TYRE', 'UANG', 'UCID', 'UDIJ',
    'UFNX', 'UGRO', 'UJSN', 'ULTJ', 'UNIC', 'UNIQ', 'UNIT', 'UNSP', 'USFI', 'VALU',
    'VICO', 'VICI', 'VIDI', 'VISI', 'VIVA', 'VKTR', 'VOKS', 'VRNA', 'VTNY', 'WAPO',
    'WEGE', 'WEHA', 'WICO', 'WIFI', 'WIIM', 'WINS', 'WMUU', 'WMPP', 'WOOD', 'WOWS',
    'WRKR', 'WSBP', 'WSKT', 'WTON', 'YELO', 'YULE', 'ZBRA', 'ZINC', 'ZONE'
]
QUALITY_STOCKS = list(dict.fromkeys(QUALITY_STOCKS))

# Sector Mapping
ENERGY_SECTOR = ['ADRO', 'PTBA', 'ITMG', 'MEDC', 'PGAS', 'ENRG', 'BUMI', 'DOID']
MINING_GOLD = ['ANTM', 'MDKA', 'BRMS']
EXPORT_SECTOR = ['ANTM', 'INCO', 'TINS', 'HRUM', 'CPIN', 'JPFA', 'MAIN']

SECTOR_MAPPING = {
    'ENERGY': ENERGY_SECTOR + ['BYAN', 'INDY', 'HRUM', 'ARTI'],
    'MINING': MINING_GOLD + ['INCO', 'TINS', 'CITA', 'DKFT'],
    'FINANCE': ['BBCA', 'BBRI', 'BMRI', 'BBNI', 'PNBN', 'BJBR', 'BJTM', 'NISP', 'BDMN', 'BNLI', 'BNGA', 'BNII', 'BSIM'],
    'PROPERTY': ['PWON', 'BSDE', 'LPKR', 'CTRA', 'SMRA', 'ASRI', 'DILD', 'MDLN', 'ELTY', 'BIPP', 'BKSL', 'MTLA', 'MAPI'],
    'CONSUMER': ['UNVR', 'ICBP', 'INDF', 'GGRM', 'HMSP', 'KLBF', 'MYOR', 'SIDO', 'ULTJ', 'DLTA', 'MLBI', 'TCID', 'ROTI', 'SKBM'],
    'INFRASTRUCTURE': ['TLKM', 'JSMR', 'TOWR', 'TBIG', 'WIKA', 'PTPP', 'WSKT', 'ADHI', 'TOTL', 'ACST'],
    'INDUSTRIAL': ['ASII', 'GJTL', 'AUTO', 'BRAM', 'INDS', 'BOLT', 'IMP', 'PRAS', 'PBRX', 'MPMX'],
    'TRADE': ['MAPI', 'ACES', 'RALS', 'LPPF', 'ERAA', 'MAPB', 'SONA', 'CSAP', 'MIDI', 'MFMI'],
    'AGRICULTURE': ['AALI', 'LSIP', 'SGRO', 'BWPT', 'SMAR', 'DSNG', 'JAWA'],
    'HEALTHCARE': ['MIKA', 'SILO', 'KLBF', 'KAEF', 'INAF', 'DVLA', 'TSPC', 'MERK', 'SCPI']
}

def get_sector(symbol):
    for sector, stocks in SECTOR_MAPPING.items():
        if symbol in stocks:
            return sector
    return 'OTHER'

# Global Indices
GLOBAL_INDICES = {
    "IHSG": "^JKSE",
    "DOWJONES": "^DJI",
    "USDIDR": "IDR=X",
    "OIL": "CL=F",
    "GOLD": "GC=F"
}

# =============================================================================
# 2. FEE CONFIGURATION
# =============================================================================

class FeeConfig:
    BROKER_FEE_BUY = 0.0015
    BROKER_FEE_SELL = 0.0025
    EXCHANGE_FEE = 0.0001

    SLIPPAGE_BUY_MIN = 0.0005
    SLIPPAGE_BUY_MAX = 0.002
    SLIPPAGE_SELL_MIN = 0.001
    SLIPPAGE_SELL_MAX = 0.003

    SLIPPAGE_MODE = 'random'
    MIN_FEE = 0

    @classmethod
    def get_slippage_buy(cls):
        if cls.SLIPPAGE_MODE == 'random':
            return random.uniform(cls.SLIPPAGE_BUY_MIN, cls.SLIPPAGE_BUY_MAX)
        else:
            return 0.001

    @classmethod
    def get_slippage_sell(cls):
        if cls.SLIPPAGE_MODE == 'random':
            return random.uniform(cls.SLIPPAGE_SELL_MIN, cls.SLIPPAGE_SELL_MAX)
        else:
            return 0.002

    @classmethod
    def calculate_buy_cost(cls, amount):
        broker = amount * cls.BROKER_FEE_BUY
        exchange = amount * cls.EXCHANGE_FEE
        slippage = amount * cls.get_slippage_buy()
        total = broker + exchange + slippage
        return max(total, cls.MIN_FEE)

    @classmethod
    def calculate_sell_cost(cls, amount):
        broker = amount * cls.BROKER_FEE_SELL
        exchange = amount * cls.EXCHANGE_FEE
        slippage = amount * cls.get_slippage_sell()
        total = broker + exchange + slippage
        return max(total, cls.MIN_FEE)

    @classmethod
    def calculate_round_trip_cost(cls, buy_amount, sell_amount):
        buy_cost = cls.calculate_buy_cost(buy_amount)
        sell_cost = cls.calculate_sell_cost(sell_amount)
        return buy_cost + sell_cost

    @classmethod
    def adjust_return_for_fee(cls, entry_price, exit_price, lot=1):
        buy_amount = entry_price * 100 * lot
        sell_amount = exit_price * 100 * lot
        total_cost = cls.calculate_round_trip_cost(buy_amount, sell_amount)
        net_profit = (sell_amount - buy_amount) - total_cost
        net_return_pct = (net_profit / buy_amount) * 100
        return net_return_pct, total_cost

# =============================================================================
# 3. LOGGING SYSTEM
# =============================================================================

class ColabLogger:
    """Logging system untuk Colab"""

    def __init__(self, log_dir='logs'):
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)

        self.log_file = f"{log_dir}/scan_{datetime.now().strftime('%Y%m%d')}.log"

        logging.basicConfig(
            filename=self.log_file,
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
        self.logger = logging.getLogger(__name__)

    def info(self, message):
        print(f"ℹ️ {message}")
        self.logger.info(message)

    def success(self, message):
        print(f"✅ {message}")
        self.logger.info(f"SUCCESS: {message}")

    def warning(self, message):
        print(f"⚠️ {message}")
        self.logger.warning(message)

    def error(self, message):
        print(f"❌ {message}")
        self.logger.error(message)

    def debug(self, message):
        print(f"🔍 {message}")
        self.logger.debug(message)

# =============================================================================
# 4. MEMORY MANAGEMENT
# =============================================================================

class MemoryManager:
    """Memory management untuk Colab"""

    def __init__(self, threshold_mb=1000):
        self.threshold_mb = threshold_mb
        self.initialized = False

    def get_memory_usage(self):
        """Estimasi memory usage"""
        try:
            import psutil
            process = psutil.Process()
            memory_info = process.memory_info()
            memory_mb = memory_info.rss / 1024 / 1024
            return memory_mb
        except:
            return 0

    def clear_memory(self, force=False):
        """Bersihkan memory"""
        gc.collect()

        memory_mb = self.get_memory_usage()
        if force or memory_mb > self.threshold_mb:
            print(f"🧹 Memory: {memory_mb:.0f}MB > threshold, cleaning...")

            if 'stocks_data' in globals():
                del globals()['stocks_data']
            if 'signals' in globals():
                del globals()['signals']

            gc.collect()

            new_memory = self.get_memory_usage()
            print(f"   Freed: {memory_mb - new_memory:.0f}MB, Now: {new_memory:.0f}MB")

    def log_memory(self, label=""):
        memory_mb = self.get_memory_usage()
        if memory_mb > 0:
            print(f"📊 Memory {label}: {memory_mb:.0f}MB")

# =============================================================================
# 5. KEEP ALIVE
# =============================================================================

class KeepAlive:
    """Mencegah Colab disconnect"""

    def __init__(self, interval=60):
        self.interval = interval
        self.running = False
        self.thread = None

    def _keep_alive(self):
        while self.running:
            time.sleep(self.interval)
            print(f"⏱️ Keep-alive: {datetime.now().strftime('%H:%M:%S')}")

    def start(self):
        if not self.running:
            self.running = True
            self.thread = threading.Thread(target=self._keep_alive, daemon=True)
            self.thread.start()
            print(f"🟢 Keep-alive started (interval {self.interval}s)")

    def stop(self):
        self.running = False
        print("🔴 Keep-alive stopped")

# =============================================================================
# 6. IDX DATA VALIDATOR (FIXED - VALIDASI LONGGAR)
# =============================================================================

class IDXDataValidator:
    """Validasi data khusus IDX - FIXED: validasi longgar"""

    def __init__(self):
        self.validation_stats = {
            'total': 0,
            'passed': 0,
            'failed': 0,
            'warnings': 0,
            'skipped': 0
        }
        self.MAX_REASONABLE_PRICE = 50_000_000  # Rp 50 juta
        self.MAX_DAILY_RETURN = 5.0  # 500% (longgar)
        self.CRITICAL_DAILY_RETURN = 10.0  # 1000% (sangat longgar)

    def is_index(self, symbol):
        """Cek apakah symbol adalah index (bukan saham)"""
        index_symbols = ['^JKSE', '^DJI', '^IXIC', '^GSPC', 'IHSG', 'LQ45', 'IDX30']
        if symbol in index_symbols:
            return True
        if symbol.startswith('^'):
            return True
        return False

    def validate(self, symbol, df):
        """Validasi dataframe untuk IDX - VERSI LONGGAR"""
        self.validation_stats['total'] += 1

        # CEK APAKAH INI INDEX
        if self.is_index(symbol):
            self.validation_stats['skipped'] += 1
            self.validation_stats['passed'] += 1
            return True, "Index - skipped validation", df

        # CEK DATA KOSONG
        if df is None or df.empty:
            self.validation_stats['failed'] += 1
            return False, "Empty data", None

        warnings_list = []

        # VALIDASI SEDERHANA - HANYA CEK HAL-HAL KRITIS
        try:
            last_close = float(df['Close'].iloc[-1])

            # Cek harga wajar (sangat longgar)
            if last_close > self.MAX_REASONABLE_PRICE:
                warnings_list.append(f"Harga tinggi: Rp {last_close:,.0f}")

            # Cek data korup (sangat longgar)
            daily_returns = df['Close'].pct_change().abs()
            if (daily_returns > self.CRITICAL_DAILY_RETURN).any():
                self.validation_stats['failed'] += 1
                return False, "Data sangat korup (daily return >1000%)", None
            elif (daily_returns > self.MAX_DAILY_RETURN).any():
                warnings_list.append(f"Volatilitas tinggi (>500%)")
                self.validation_stats['warnings'] += 1

            # Lolos validasi
            self.validation_stats['passed'] += 1

            if warnings_list:
                return True, f"OK dengan warning: {', '.join(warnings_list)}", df
            else:
                return True, "OK", df

        except Exception as e:
            # Jika error, coba beri kesempatan
            self.validation_stats['passed'] += 1
            return True, f"Validasi longgar: {str(e)[:50]}", df

    def print_stats(self):
        """Print statistik validasi"""
        print(f"\n📊 Validasi Data IDX:")
        print(f"   Total: {self.validation_stats['total']}")
        print(f"   ✅ Passed: {self.validation_stats['passed']}")
        print(f"   ⏭️  Skipped (Index): {self.validation_stats['skipped']}")
        print(f"   ❌ Failed: {self.validation_stats['failed']}")
        print(f"   ⚠️  Warnings: {self.validation_stats['warnings']}")

# =============================================================================
# 7. STATE MANAGER
# =============================================================================

class ColabStateManager:
    """State management untuk Colab dengan Google Drive"""

    def __init__(self, base_dir='/content/drive/MyDrive/idx_scanner_state'):
        self.base_dir = base_dir
        self.state_file = f"{base_dir}/latest_state.pkl"
        self.history_dir = f"{base_dir}/history"
        self.metadata_file = f"{base_dir}/metadata.json"

        try:
            drive.mount('/content/drive')
        except:
            pass

        os.makedirs(base_dir, exist_ok=True)
        os.makedirs(self.history_dir, exist_ok=True)

        self.logger = ColabLogger()

    def _get_state_id(self, data):
        timestamp = datetime.now().isoformat()
        hash_input = f"{timestamp}_{len(data) if data else 0}"
        return hashlib.md5(hash_input.encode()).hexdigest()[:8]

    def save_state(self, data, metadata=None):
        try:
            state_id = self._get_state_id(data)
            timestamp = datetime.now()

            state_data = {
                'id': state_id,
                'timestamp': timestamp,
                'data': data,
                'metadata': metadata or {}
            }

            with open(self.state_file, 'wb') as f:
                pickle.dump(state_data, f)

            history_file = f"{self.history_dir}/state_{timestamp.strftime('%Y%m%d_%H%M%S')}_{state_id}.pkl"
            with open(history_file, 'wb') as f:
                pickle.dump(state_data, f)

            self._update_metadata({
                'last_save': timestamp.isoformat(),
                'last_id': state_id,
                'total_saves': self._get_metadata().get('total_saves', 0) + 1
            })

            self.logger.success(f"State saved: {state_id}")
            self._cleanup_old_states(keep=7)

            return True

        except Exception as e:
            self.logger.error(f"Failed to save state: {e}")
            return False

    def load_latest_state(self):
        try:
            if not os.path.exists(self.state_file):
                self.logger.warning("No previous state found")
                return None

            with open(self.state_file, 'rb') as f:
                state_data = pickle.load(f)

            self.logger.success(f"Loaded state from {state_data['timestamp']}")
            return state_data

        except Exception as e:
            self.logger.error(f"Failed to load state: {e}")
            return None

    def load_history(self, days=7):
        try:
            states = []
            cutoff = datetime.now() - timedelta(days=days)

            for filename in os.listdir(self.history_dir):
                if filename.startswith('state_') and filename.endswith('.pkl'):
                    filepath = os.path.join(self.history_dir, filename)

                    with open(filepath, 'rb') as f:
                        state = pickle.load(f)

                    state_time = datetime.fromisoformat(state['timestamp']) if isinstance(state['timestamp'], str) else state['timestamp']

                    if state_time > cutoff:
                        states.append(state)

            states.sort(key=lambda x: x['timestamp'], reverse=True)

            self.logger.info(f"Loaded {len(states)} historical states")
            return states

        except Exception as e:
            self.logger.error(f"Failed to load history: {e}")
            return []

    def _update_metadata(self, updates):
        metadata = self._get_metadata()
        metadata.update(updates)

        with open(self.metadata_file, 'w') as f:
            json.dump(metadata, f, indent=2)

    def _get_metadata(self):
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, 'r') as f:
                return json.load(f)
        return {'total_saves': 0}

    def _cleanup_old_states(self, keep=7):
        try:
            states = []
            for filename in os.listdir(self.history_dir):
                if filename.startswith('state_') and filename.endswith('.pkl'):
                    filepath = os.path.join(self.history_dir, filename)
                    mtime = os.path.getmtime(filepath)
                    states.append((filepath, mtime))

            states.sort(key=lambda x: x[1], reverse=True)

            for filepath, _ in states[keep:]:
                os.remove(filepath)

        except Exception as e:
            self.logger.error(f"Cleanup failed: {e}")

    def export_state(self, filename='state_export.pkl'):
        if os.path.exists(self.state_file):
            files.download(self.state_file)
            self.logger.info("State exported")

    def import_state(self):
        try:
            uploaded = files.upload()
            for filename in uploaded.keys():
                with open(filename, 'rb') as f:
                    state = pickle.load(f)

                with open(self.state_file, 'wb') as f:
                    pickle.dump(state, f)

                self.logger.success(f"Imported state from {filename}")
                return state

        except Exception as e:
            self.logger.error(f"Import failed: {e}")
            return None

# =============================================================================
# 8. CONFIGURATION MANAGER (DENGAN ADAPTIVE FILTER)
# =============================================================================

class ConfigManager:
    """Manajemen konfigurasi untuk semua engine"""

    DEFAULT_CONFIG = {
        'global': {
            'cache_dir': 'stock_cache_v3',
            'max_workers': 10,
            'log_level': 'INFO',
            'auto_export_sheets': False
        },
        'swing': {
            'enabled': True,
            'min_price': 50,  # Turunkan untuk modal kecil
            'max_price': 50000,
            'min_volume': 10000,  # Turunkan
            'max_spread_pct': 2.0,
            'min_avg_volume': 100000,  # Turunkan
            'min_rr': 1.0,
            'max_portfolio_risk_pct': 3.0,
            'min_ev_pct': 2.0,
            'interval': '1d',
            'period': '6mo',
            'min_history': 60,
            'rsi_period': 14,
            'ma_short': 20,
            'ma_long': 50,
            'ma200_period': 200,
            'support_period': 60,
            'atr_period': 14,
            'sl_multiplier': 1.5,
            'tp_multiplier': 2.5,
            'base_threshold': 5,
            'volume_boost': 1.5
        },
        'intraday': {
            'enabled': True,
            'min_price': 50,  # Turunkan untuk modal kecil
            'max_price': 50000,
            'min_volume': 10000,  # Turunkan
            'max_spread_pct': 2.0,
            'min_avg_volume': 100000,  # Turunkan
            'min_rr': 1.0,
            'max_portfolio_risk_pct': 3.0,
            'min_ev_pct': 2.0,
            'interval': '1h',
            'period': '1mo',
            'min_history': 30,
            'breakout_period': 10,
            'volume_breakout': 1.2,
            'ma_short': 20,
            'ma_long': 50,
            'atr_period': 14,
            'tp_multiplier': 1.2,
            'sl_multiplier': 0.8,
            'risk_per_trade_pct': 0.5,
            'max_trades_per_day': 3
        },
        'gorengan': {
            'enabled': False,
            'min_price': 50,
            'max_price': 50000,
            'min_volume': 5000,
            'max_spread_pct': 3.0,
            'min_avg_volume': 100000,  # Turunkan
            'min_rr': 1.0,
            'max_portfolio_risk_pct': 5.0,
            'min_ev_pct': 1.5,
            'interval': '1h',
            'period': '1mo',
            'min_history': 30,
            'risk_per_trade_pct': 0.3,
            'max_trades_per_day': 2
        },
        'investasi': {
            'enabled': True,
            'min_price': 50,
            'max_price': 50000,
            'min_volume': 5000,
            'max_spread_pct': 7.0,
            'min_avg_volume': 50000,
            'min_ev_pct': 1.0,
            'max_portfolio_risk_pct': 10.0,
            'max_portfolio': 5,
            'interval': '1d',
            'period': '2y',
            'min_history': 100,
            'ma200_strict': True,
            'ma50_lower': -20,
            'ma50_upper': 15
        }
    }

    def __init__(self, config_dir='config'):
        self.config_dir = config_dir
        self.config_file = f"{config_dir}/scanner_config.json"
        self.backup_dir = f"{config_dir}/backups"
        self.config = None

        os.makedirs(config_dir, exist_ok=True)
        os.makedirs(self.backup_dir, exist_ok=True)

        self.logger = ColabLogger()
        self.load_config()

    def load_config(self):
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, 'r') as f:
                    self.config = json.load(f)
                self.logger.info(f"Config loaded from {self.config_file}")
                self._merge_with_default()
            except Exception as e:
                self.logger.error(f"Failed to load config: {e}")
                self.config = self.DEFAULT_CONFIG.copy()
        else:
            self.logger.info("No config file found, using defaults")
            self.config = self.DEFAULT_CONFIG.copy()
            self.save_config()

    def _merge_with_default(self):
        for key, default_value in self.DEFAULT_CONFIG.items():
            if key not in self.config:
                self.config[key] = default_value
            elif isinstance(default_value, dict):
                for subkey, subvalue in default_value.items():
                    if subkey not in self.config[key]:
                        self.config[key][subkey] = subvalue

    def save_config(self, backup=False):
        try:
            if backup:
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                backup_file = f"{self.backup_dir}/config_{timestamp}.json"
                with open(backup_file, 'w') as f:
                    json.dump(self.config, f, indent=2)
                self.logger.info(f"Config backed up to {backup_file}")

            with open(self.config_file, 'w') as f:
                json.dump(self.config, f, indent=2)
            self.logger.info(f"Config saved to {self.config_file}")
            return True
        except Exception as e:
            self.logger.error(f"Failed to save config: {e}")
            return False

    def get_engine_config(self, engine_name):
        return self.config.get(engine_name, {})

    def get_global_config(self):
        return self.config.get('global', {})

    def update_engine_config(self, engine_name, updates):
        if engine_name in self.config:
            self.config[engine_name].update(updates)
            self.save_config(backup=True)
            return True
        return False

    def reset_to_default(self, engine_name=None):
        if engine_name:
            if engine_name in self.DEFAULT_CONFIG:
                self.config[engine_name] = self.DEFAULT_CONFIG[engine_name].copy()
                self.logger.info(f"Reset {engine_name} to default")
        else:
            self.config = self.DEFAULT_CONFIG.copy()
            self.logger.info("Reset all config to default")

        self.save_config(backup=True)

    def list_backups(self):
        backups = []
        for f in os.listdir(self.backup_dir):
            if f.startswith('config_') and f.endswith('.json'):
                filepath = os.path.join(self.backup_dir, f)
                mtime = datetime.fromtimestamp(os.path.getmtime(filepath))
                size = os.path.getsize(filepath)
                backups.append({
                    'file': f,
                    'time': mtime,
                    'size': size
                })

        return sorted(backups, key=lambda x: x['time'], reverse=True)

    def restore_backup(self, backup_file):
        try:
            backup_path = os.path.join(self.backup_dir, backup_file)
            with open(backup_path, 'r') as f:
                backup_config = json.load(f)

            self.save_config(backup=True)
            self.config = backup_config
            self.save_config()
            self.logger.info(f"Restored config from {backup_file}")
            return True
        except Exception as e:
            self.logger.error(f"Failed to restore backup: {e}")
            return False

    def print_config(self, engine_name=None):
        print("\n" + "="*80)
        print("📋 CURRENT CONFIGURATION")
        print("="*80)

        if engine_name:
            if engine_name in self.config:
                print(f"\n🔧 {engine_name.upper()} ENGINE:")
                for key, value in self.config[engine_name].items():
                    print(f"   {key}: {value}")
        else:
            for section, values in self.config.items():
                print(f"\n📁 {section.upper()}:")
                if isinstance(values, dict):
                    for key, value in values.items():
                        print(f"   {key}: {value}")
                else:
                    print(f"   {values}")

# =============================================================================
# 9. RTI SCRAPER
# =============================================================================

class RTIScraper:
    """Scraper untuk data fundamental dari RTI (HANYA UNTUK DISPLAY)"""

    def __init__(self):
        self.logger = ColabLogger()
        self.cache_dir = 'rti_cache'
        self.cache_duration = 7 * 24 * 3600
        os.makedirs(self.cache_dir, exist_ok=True)

        self.stats = {
            'total': 0,
            'cache_hits': 0,
            'success': 0,
            'failed': 0
        }

    def _get_cache_key(self, symbol):
        return f"{symbol}_rti.json"

    def _load_from_cache(self, symbol):
        cache_file = f"{self.cache_dir}/{self._get_cache_key(symbol)}"
        if os.path.exists(cache_file):
            try:
                file_time = os.path.getmtime(cache_file)
                if time.time() - file_time < self.cache_duration:
                    with open(cache_file, 'r') as f:
                        data = json.load(f)
                    self.stats['cache_hits'] += 1
                    return data
            except:
                pass
        return None

    def _save_to_cache(self, symbol, data):
        cache_file = f"{self.cache_dir}/{self._get_cache_key(symbol)}"
        try:
            with open(cache_file, 'w') as f:
                json.dump(data, f)
        except:
            pass

    def scrape(self, symbol):
        self.stats['total'] += 1

        cached = self._load_from_cache(symbol)
        if cached:
            return cached

        try:
            url = f"https://www.rti.co.id/saham/{symbol}"
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                self.stats['failed'] += 1
                return self._get_fallback_data(symbol)

            soup = BeautifulSoup(response.text, 'html.parser')

            data = {
                'symbol': symbol,
                'timestamp': datetime.now().isoformat(),
                'per': None,
                'pbv': None,
                'roe': None,
                'der': None,
                'market_cap': None,
                'dividend_yield': None,
                'source': 'rti',
                'note': 'Hanya untuk display'
            }

            try:
                per_elem = soup.find('div', {'data-label': 'PER'})
                if per_elem:
                    per_text = per_elem.find('div', class_='value')
                    if per_text:
                        data['per'] = self._parse_number(per_text.text)
            except:
                pass

            try:
                pbv_elem = soup.find('div', {'data-label': 'PBV'})
                if pbv_elem:
                    pbv_text = pbv_elem.find('div', class_='value')
                    if pbv_text:
                        data['pbv'] = self._parse_number(pbv_text.text)
            except:
                pass

            try:
                mc_elem = soup.find('div', {'data-label': 'Market Cap'})
                if mc_elem:
                    mc_text = mc_elem.find('div', class_='value')
                    if mc_text:
                        data['market_cap'] = self._parse_market_cap(mc_text.text)
            except:
                pass

            if all(v is None for v in [data['per'], data['pbv'], data['market_cap']]):
                data = self._get_fallback_data(symbol)
                data['source'] = 'fallback'
            else:
                self.stats['success'] += 1

            self._save_to_cache(symbol, data)
            return data

        except Exception as e:
            self.logger.debug(f"RTI scrape failed for {symbol}: {e}")
            self.stats['failed'] += 1
            return self._get_fallback_data(symbol)

    def _parse_number(self, text):
        try:
            clean = text.strip().replace(',', '').replace(' ', '')
            return float(clean)
        except:
            return None

    def _parse_market_cap(self, text):
        try:
            clean = text.strip().replace(',', '').replace(' ', '')
            if 'T' in clean:
                return float(clean.replace('T', '')) * 1_000_000_000_000
            elif 'M' in clean:
                return float(clean.replace('M', '')) * 1_000_000_000
            else:
                return float(clean)
        except:
            return None

    def _get_fallback_data(self, symbol):
        return {
            'symbol': symbol,
            'timestamp': datetime.now().isoformat(),
            'per': 'N/A',
            'pbv': 'N/A',
            'roe': 'N/A',
            'der': 'N/A',
            'market_cap': 'N/A',
            'dividend_yield': 'N/A',
            'source': 'fallback',
            'note': 'Data tidak tersedia'
        }

    def print_stats(self):
        print(f"\n📊 RTI Scraper Statistics:")
        print(f"   Total attempts: {self.stats['total']}")
        print(f"   ✅ Success: {self.stats['success']}")
        print(f"   📦 Cache hits: {self.stats['cache_hits']}")
        print(f"   ❌ Failed: {self.stats['failed']}")

# =============================================================================
# 10. GOOGLE SHEETS EXPORTER (DENGAN URL TETAP MUNCUL)
# =============================================================================

class GoogleSheetsExporter:
    """Export hasil scan ke Google Sheets - 1 sheet per engine"""

    def __init__(self):
        self.logger = ColabLogger()
        self.initialized = False
        self.spreadsheet = None
        self._url_printed = False

        # Template header per engine
        self.HEADERS = {
            'SWING ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'RSI',
                'Support', 'Resistance', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Volume', 'Inflow', 'Acc'
            ],
            'INTRADAY LIQUID ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'Volume Ratio',
                'Body Ratio', 'Upper Wick', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Inflow', 'Acc'
            ],
            'INTRADAY GORENGAN ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'Volume Spike',
                'Turnover', 'Day Change %', 'Stop Loss', 'Take Profit',
                'R/R', 'Prob Up', 'EV %', 'Score', 'Inflow', 'Acc'
            ],
            'INVESTASI ENGINE': [
                'Date', 'Timestamp', 'Symbol', 'Sector', 'Price', 'To MA50',
                'MA50', 'MA200', 'PER', 'PBV', 'Market Cap',
                'Stop Loss', 'Risk %', 'Lot', 'Cost', 'Quality', 'MA200+'
            ]
        }

    def _init_sheets(self):
        """Initialize Google Sheets API"""
        try:
            auth.authenticate_user()
            creds, _ = default()
            self.gc = gspread.authorize(creds)
            self.initialized = True
            return True
        except Exception as e:
            self.logger.error(f"Failed to initialize Google Sheets: {e}")
            return False

    def _get_or_create_spreadsheet(self, sheet_name):
        """Get atau create spreadsheet utama"""
        try:
            self.spreadsheet = self.gc.open(sheet_name)
            self.logger.info(f"Opened existing spreadsheet: {sheet_name}")
        except:
            self.spreadsheet = self.gc.create(sheet_name)
            self.logger.info(f"Created new spreadsheet: {sheet_name}")

            # Share dengan user (opsional)
            try:
                email = input("\n📧 Share spreadsheet dengan email (optional, tekan Enter untuk skip): ").strip()
                if email:
                    self.spreadsheet.share(email, perm_type='user', role='writer')
                    print(f"✅ Shared with {email}")
            except:
                pass

        return self.spreadsheet

    def _get_or_create_engine_sheet(self, engine_name):
        """Get atau create sheet untuk engine tertentu (1 sheet permanen)"""
        sheet_title = engine_name.replace(' ', '_').replace('ENGINE', '').strip()[:100]

        try:
            worksheet = self.spreadsheet.worksheet(sheet_title)
            self.logger.info(f"Opened existing sheet: {sheet_title}")

            existing_headers = worksheet.row_values(1)
            if not existing_headers:
                headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
                worksheet.append_row(headers)

        except:
            worksheet = self.spreadsheet.add_worksheet(
                title=sheet_title,
                rows=10000,
                cols=30
            )
            self.logger.info(f"Created new sheet: {sheet_title}")

            headers = self.HEADERS.get(engine_name, ['Date', 'Timestamp', 'Symbol', 'Signal'])
            worksheet.append_row(headers)

        return worksheet

    def _format_signal_row(self, signal, engine_name):
        """Format signal berdasarkan engine"""
        today = datetime.now().strftime('%Y-%m-%d')
        timestamp = datetime.now().strftime('%H:%M:%S')

        if engine_name == 'SWING ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('RSI', '-'),
                signal.get('Support', '-'),
                signal.get('Resistance', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Volume', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INTRADAY LIQUID ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('Volume', '-'),
                signal.get('Body_Ratio', '-'),
                signal.get('Upper_Wick', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INTRADAY GORENGAN ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal.get('Volume_Spike', '-'),
                signal.get('Turnover', '-'),
                signal.get('Day_Change', '-'),
                signal.get('Stop_Loss', '-'),
                signal.get('Take_Profit', '-'),
                signal.get('R/R', '-'),
                f"{signal.get('Prob_Up', 0):.0%}" if signal.get('Prob_Up') else '-',
                signal.get('EV_Pct', '-'),
                signal.get('Score', '-'),
                signal.get('Inflow', '-'),
                signal.get('Acc', '-')
            ]

        elif engine_name == 'INVESTASI ENGINE':
            return [
                today,
                timestamp,
                signal['Symbol'],
                signal['Sector'],
                signal['Price'],
                signal['To_MA50'],
                signal.get('MA50', '-'),
                signal.get('MA200', '-'),
                signal.get('PER', 'N/A'),
                signal.get('PBV', 'N/A'),
                signal.get('MarketCap', 'N/A'),
                signal['Stop_Loss'],
                f"{signal.get('Risk%', 0)}%",
                signal['Lot'],
                signal['Cost'],
                signal.get('Quality', '-'),
                "⭐" if signal.get('MA200_Bonus', 0) else "-"
            ]

        else:
            return [today, timestamp, signal['Symbol'], str(signal)]

    def ensure_spreadsheet_exists(self):
        """Pastikan spreadsheet sudah ada dan tampilkan URL (meski 0 sinyal)"""
        if not self.initialized:
            if not self._init_sheets():
                return False

        if not self.spreadsheet:
            sheet_name = "IDX_Scanner_All_Engines"
            self._get_or_create_spreadsheet(sheet_name)

        if not self._url_printed:
            print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
            self._url_printed = True

        return True

    def export_signals(self, signals, engine_name, modal, auto_confirm=True):
        """Export sinyal ke Google Sheets"""
        if not signals:
            self.logger.info(f"No signals to export for {engine_name}")
            return False

        if not auto_confirm:
            choice = input(f"\n📊 Export {engine_name} signals ke Google Sheets? (y/n): ").strip().lower()
            if choice != 'y':
                print("✅ Export skipped")
                return False

        if not self.initialized:
            if not self._init_sheets():
                return False

        try:
            if not self.spreadsheet:
                sheet_name = "IDX_Scanner_All_Engines"
                self._get_or_create_spreadsheet(sheet_name)

            worksheet = self._get_or_create_engine_sheet(engine_name)

            for signal in signals:
                row = self._format_signal_row(signal, engine_name)
                worksheet.append_row(row)

            self.logger.success(f"Exported {len(signals)} signals to {engine_name} sheet")

            if not self._url_printed:
                print(f"\n📊 Google Sheets URL: {self.spreadsheet.url}")
                self._url_printed = True

            return True

        except Exception as e:
            self.logger.error(f"Failed to export to Google Sheets: {e}")
            return False

# =============================================================================
# 11. MARKET REGIME DETECTOR (FIXED - DENGAN .flatten())
# =============================================================================

class MarketRegimeDetector:
    """Deteksi kondisi pasar menggunakan IHSG - FIXED"""

    def __init__(self, global_fetcher):
        self.global_fetcher = global_fetcher
        self.logger = ColabLogger()

    def detect_trend_strength(self, period=20):
        """Deteksi kekuatan trend IHSG menggunakan ADX - FIXED"""
        try:
            if 'IHSG' not in self.global_fetcher.data:
                return "NO_IHSG_DATA", 0

            df = self.global_fetcher.data['IHSG']
            if len(df) < period + 10:
                return "INSUFFICIENT_DATA", 0

            # FIX: Gunakan .flatten() untuk mengubah array 2D ke 1D
            high = df['High'].values.flatten()
            low = df['Low'].values.flatten()
            close = df['Close'].values.flatten()

            # True Range
            tr = np.maximum(high[1:] - low[1:],
                           np.abs(high[1:] - close[:-1]),
                           np.abs(low[1:] - close[:-1]))
            tr = np.concatenate([[tr[0]], tr])
            atr = pd.Series(tr).rolling(window=period).mean().values

            # Directional Movement
            up_move = high[1:] - high[:-1]
            down_move = low[:-1] - low[1:]

            plus_dm = np.zeros_like(up_move)
            minus_dm = np.zeros_like(down_move)

            for i in range(len(up_move)):
                if up_move[i] > down_move[i] and up_move[i] > 0:
                    plus_dm[i] = up_move[i]
                if down_move[i] > up_move[i] and down_move[i] > 0:
                    minus_dm[i] = down_move[i]

            plus_dm = np.concatenate([[0], plus_dm])
            minus_dm = np.concatenate([[0], minus_dm])

            plus_di = 100 * (pd.Series(plus_dm).rolling(window=period).mean() / (atr + 1e-10))
            minus_di = 100 * (pd.Series(minus_dm).rolling(window=period).mean() / (atr + 1e-10))

            dx = 100 * np.abs(plus_di - minus_di) / (plus_di + minus_di + 1e-10)
            adx = dx.rolling(window=period).mean()

            current_adx = float(adx.iloc[-1]) if not pd.isna(adx.iloc[-1]) else 0

            if current_adx > 25:
                return "TRENDING", current_adx
            elif current_adx > 20:
                return "WEAK_TREND", current_adx
            else:
                return "RANGING", current_adx

        except Exception as e:
            self.logger.error(f"ADX calculation error: {e}")
            return "ERROR", 0

    def detect_volatility_regime(self, period=20):
        """Deteksi regime volatilitas IHSG"""
        try:
            if 'IHSG' not in self.global_fetcher.data:
                return "NO_IHSG_DATA", 1

            df = self.global_fetcher.data['IHSG']
            if len(df) < period + 10:
                return "INSUFFICIENT_DATA", 1

            returns = df['Close'].pct_change().dropna()
            if len(returns) < period:
                return "INSUFFICIENT_DATA", 1

            current_vol = returns.tail(period).std() * np.sqrt(252)
            hist_vol = returns.std() * np.sqrt(252)

            vol_ratio = current_vol / hist_vol if hist_vol > 0 else 1

            if vol_ratio > 1.5:
                return "HIGH_VOL", vol_ratio
            elif vol_ratio > 1.2:
                return "MEDIUM_VOL", vol_ratio
            else:
                return "LOW_VOL", vol_ratio

        except Exception as e:
            return "ERROR", 1

    def detect_sector_rotation(self, stocks_data):
        """Deteksi sektor mana yang outperformed"""
        sector_returns = defaultdict(list)

        for symbol, df in stocks_data.items():
            if len(df) < 20:
                continue

            sector = get_sector(symbol)
            try:
                ret_20d = (df['Close'].iloc[-1] / df['Close'].iloc[-20] - 1) * 100
                sector_returns[sector].append(ret_20d)
            except:
                continue

        sector_performance = {}
        for sector, returns in sector_returns.items():
            if returns:
                sector_performance[sector] = np.mean(returns)

        top_sectors = sorted(sector_performance.items(), key=lambda x: -x[1])[:3]

        return top_sectors, sector_performance

# =============================================================================
# 12. QUALITY ESTIMATOR
# =============================================================================

class QualityEstimator:
    """Estimasi kualitas saham tanpa fundamental"""

    def __init__(self):
        self.WEIGHTS = {
            'likuiditas': 0.30,
            'volatilitas': 0.20,
            'trend': 0.30,
            'indeks': 0.20
        }

    def estimate(self, symbol, df):
        """Estimasi kualitas saham"""
        score = 0
        details = {}

        # 1. Likuiditas
        avg_volume = df['Volume'].tail(60).mean()
        if avg_volume > 2_000_000:
            liq_score = 100
        elif avg_volume > 1_000_000:
            liq_score = 80
        elif avg_volume > 500_000:
            liq_score = 60
        elif avg_volume > 200_000:
            liq_score = 40
        else:
            liq_score = 20

        score += liq_score * self.WEIGHTS['likuiditas']
        details['likuiditas'] = liq_score

        # 2. Volatilitas
        returns = df['Close'].pct_change().dropna()
        volatility = returns.std() * np.sqrt(252)

        if volatility < 0.20:
            vol_score = 100
        elif volatility < 0.30:
            vol_score = 80
        elif volatility < 0.40:
            vol_score = 60
        elif volatility < 0.60:
            vol_score = 40
        else:
            vol_score = 20

        score += vol_score * self.WEIGHTS['volatilitas']
        details['volatilitas'] = vol_score

        # 3. Trend
        if len(df) >= 200:
            price_200d_ago = df['Close'].iloc[-200]
            price_now = df['Close'].iloc[-1]
            trend_return = (price_now / price_200d_ago - 1) * 100

            if trend_return > 50:
                trend_score = 100
            elif trend_return > 20:
                trend_score = 80
            elif trend_return > 0:
                trend_score = 60
            elif trend_return > -20:
                trend_score = 40
            else:
                trend_score = 20
        else:
            trend_score = 50

        score += trend_score * self.WEIGHTS['trend']
        details['trend'] = trend_score

        # 4. Indeks
        if symbol in QUALITY_STOCKS[:50]:
            idx_score = 100
        elif symbol in QUALITY_STOCKS[:100]:
            idx_score = 80
        elif symbol in QUALITY_STOCKS:
            idx_score = 60
        else:
            idx_score = 40

        score += idx_score * self.WEIGHTS['indeks']
        details['indeks'] = idx_score

        if score >= 80:
            category = "⭐ BLUE CHIP"
        elif score >= 65:
            category = "🌟 QUALITY"
        elif score >= 50:
            category = "📊 MID CAP"
        elif score >= 35:
            category = "📈 SMALL CAP"
        else:
            category = "⚠️ SPECULATIVE"

        return {
            'score': round(score, 1),
            'category': category,
            'details': details
        }

# =============================================================================
# 13. ADAPTIVE TIMEOUT FETCHER
# =============================================================================

class AdaptiveTimeoutFetcher:
    """Fetcher dengan timeout adaptif"""

    def __init__(self):
        self.timeouts = [5, 8, 10, 15]
        self.stats = {'success': 0, 'timeout_1': 0, 'timeout_2': 0, 'timeout_3': 0, 'failed': 0}

    def fetch_with_adaptive_timeout(self, symbol, config):
        for attempt, timeout in enumerate(self.timeouts):
            try:
                ticker = f"{symbol}.JK"

                df = yf.download(
                    ticker,
                    period=config.PERIOD,
                    interval=config.INTERVAL,
                    auto_adjust=True,
                    progress=False,
                    timeout=timeout
                )

                if not df.empty:
                    self.stats['success'] += 1
                    if attempt == 0:
                        self.stats['timeout_1'] += 1
                    elif attempt == 1:
                        self.stats['timeout_2'] += 1
                    elif attempt == 2:
                        self.stats['timeout_3'] += 1

                    if isinstance(df.columns, pd.MultiIndex):
                        df.columns = df.columns.get_level_values(0)

                    return df

            except Exception as e:
                if attempt == len(self.timeouts) - 1:
                    self.stats['failed'] += 1
                    return None
                time.sleep(1)

        self.stats['failed'] += 1
        return None

    def print_stats(self):
        print(f"\n📊 Adaptive Timeout Statistics:")
        print(f"   ✅ Success: {self.stats['success']}")
        print(f"   ⏱️  Timeout 5s: {self.stats['timeout_1']}")
        print(f"   ⏱️  Timeout 8s: {self.stats['timeout_2']}")
        print(f"   ⏱️  Timeout 10s: {self.stats['timeout_3']}")
        print(f"   ❌ Failed: {self.stats['failed']}")

# =============================================================================
# 14. YFINANCE FETCHER
# =============================================================================

class YFinanceFetcher:
    """Fetcher untuk yfinance dengan adaptive timeout"""

    def __init__(self):
        self.logger = ColabLogger()
        self.adaptive_fetcher = AdaptiveTimeoutFetcher()
        self.stats = {
            'total': 0,
            'success': 0,
            'failed': 0,
            'filtered': 0,
            'cached': 0
        }
        self.cache_dir = 'yfinance_cache'
        os.makedirs(self.cache_dir, exist_ok=True)

    def _get_cache_key(self, symbol, config):
        today = datetime.now().strftime("%Y-%m-%d")
        period = config.PERIOD
        interval = config.INTERVAL
        return f"{symbol}_{period}_{interval}_{today}.pkl"

    def _load_from_cache(self, cache_key):
        cache_file = f"{self.cache_dir}/{cache_key}"
        if os.path.exists(cache_file):
            try:
                file_time = datetime.fromtimestamp(os.path.getmtime(cache_file))
                if datetime.now() - file_time < timedelta(hours=6):
                    with open(cache_file, 'rb') as f:
                        df = pickle.load(f)
                    self.stats['cached'] += 1
                    return df
            except:
                pass
        return None

    def _save_to_cache(self, cache_key, data):
        cache_file = f"{self.cache_dir}/{cache_key}"
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump(data, f)
        except:
            pass

    def _apply_filters(self, df, config):
        try:
            if hasattr(config, 'MIN_PRICE'):
                last_close = float(df['Close'].iloc[-1])
                if last_close < config.MIN_PRICE or (hasattr(config, 'MAX_PRICE') and last_close > config.MAX_PRICE):
                    return None

            if hasattr(config, 'MIN_VOLUME'):
                last_volume = int(df['Volume'].iloc[-1])
                if last_volume < config.MIN_VOLUME:
                    return None

            return df

        except Exception:
            return None

    def fetch(self, symbol, config):
        self.stats['total'] += 1

        cache_key = self._get_cache_key(symbol, config)
        cached_df = self._load_from_cache(cache_key)
        if cached_df is not None:
            filtered_df = self._apply_filters(cached_df, config)
            if filtered_df is not None:
                self.stats['success'] += 1
                return filtered_df

        df = self.adaptive_fetcher.fetch_with_adaptive_timeout(symbol, config)
        if df is None:
            self.stats['failed'] += 1
            return None

        if len(df) < config.MIN_HISTORY:
            self.stats['filtered'] += 1
            return None

        filtered_df = self._apply_filters(df, config)
        if filtered_df is None:
            self.stats['filtered'] += 1
            return None

        self._save_to_cache(cache_key, filtered_df)

        self.stats['success'] += 1
        return filtered_df

    def print_stats(self):
        print(f"\n📊 YFinance Statistics:")
        print(f"   Total attempts: {self.stats['total']}")
        print(f"   ✅ Success: {self.stats['success']}")
        print(f"   📦 From cache: {self.stats['cached']}")
        print(f"   ❌ Failed: {self.stats['failed']}")
        print(f"   🔍 Filtered: {self.stats['filtered']}")
        self.adaptive_fetcher.print_stats()

# =============================================================================
# 15. OPTIMIZED DATA FETCHER (FIXED - VALIDASI MINIMAL)
# =============================================================================

class OptimizedDataFetcher:
    """Data fetcher dengan YFinance - FIXED: validasi minimal"""

    def __init__(self, cache_dir='stock_cache_v3', max_workers=10):
        self.cache_dir = cache_dir
        self.max_workers = max_workers
        os.makedirs(cache_dir, exist_ok=True)

        self.yfinance = YFinanceFetcher()
        self.validator = IDXDataValidator()
        self.logger = ColabLogger()
        self.memory_manager = MemoryManager()

        self.stats = {
            'total': 0,
            'success': 0,
            'cached': 0,
            'failed': 0,
            'filtered': 0
        }

    def _apply_filters(self, df, config, symbol):
        """Apply filter - VERSI MINIMAL"""
        try:
            # HANYA filter yang benar-benar penting
            if hasattr(config, 'MIN_PRICE'):
                last_close = float(df['Close'].iloc[-1])
                if last_close < config.MIN_PRICE:
                    return None

            if hasattr(config, 'MAX_PRICE'):
                last_close = float(df['Close'].iloc[-1])
                if last_close > config.MAX_PRICE:
                    return None

            # Untuk intraday, filter sesi
            if hasattr(config, 'JKT_TZ') and config.MODE == 'intraday':
                df_filtered = self._filter_jakarta_sessions(df, config)
                if not df_filtered.empty and len(df_filtered) >= 10:
                    df = df_filtered
                else:
                    return None

            return df

        except Exception as e:
            return None

    def _filter_jakarta_sessions(self, df, config):
        """Filter untuk sesi Jakarta"""
        out = df.copy()
        if not isinstance(out.index, pd.DatetimeIndex):
            return out

        if out.index.tz is None:
            out.index = out.index.tz_localize(config.JKT_TZ)
        else:
            out.index = out.index.tz_convert(config.JKT_TZ)

        sess1 = out.between_time(config.SESSION1_START, config.SESSION1_END, inclusive="both")
        sess2 = out.between_time(config.SESSION2_START, config.SESSION2_END, inclusive="both")

        if not sess1.empty or not sess2.empty:
            out = pd.concat([sess1, sess2]).sort_index()

        if isinstance(out.index, pd.DatetimeIndex):
            out = out[out.index.dayofweek < 5]

        return out

    def fetch_single(self, symbol, config):
        """Fetch single symbol dengan validasi minimal"""
        self.stats['total'] += 1

        # Fetch dari YFinance
        df = self.yfinance.fetch(symbol, config)
        if df is None:
            self.stats['failed'] += 1
            return None

        # Validasi MINIMAL (hanya cek data kosong)
        if df is None or df.empty:
            self.stats['failed'] += 1
            return None

        # Apply filters minimal
        filtered_df = self._apply_filters(df, config, symbol)
        if filtered_df is None:
            self.stats['filtered'] += 1
            return None

        self.stats['success'] += 1
        return filtered_df

    def fetch_parallel(self, symbols, config):
        """Fetch multiple symbols in parallel"""
        self.logger.info(f"Fetching {len(symbols)} stocks with {self.max_workers} workers...")

        results = {}
        start_time = time.time()

        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            future_to_symbol = {
                executor.submit(self.fetch_single, symbol, config): symbol
                for symbol in symbols
            }

            for i, future in enumerate(as_completed(future_to_symbol)):
                symbol = future_to_symbol[future]
                try:
                    df = future.result()
                    if df is not None:
                        results[symbol] = df

                    if (i + 1) % 100 == 0:
                        elapsed = time.time() - start_time
                        rate = (i + 1) / elapsed if elapsed > 0 else 0
                        self.logger.info(
                            f"Progress: {i+1}/{len(symbols)} - "
                            f"{len(results)} OK - "
                            f"{rate:.1f} stocks/sec"
                        )

                except Exception as e:
                    pass

        elapsed = time.time() - start_time
        self.logger.success(f"Completed in {elapsed:.1f}s - {len(results)} OK, {len(symbols) - len(results)} failed")

        return results

    def print_stats(self):
        """Print detailed statistics"""
        print("\n" + "="*50)
        print("📊 FETCHER STATISTICS")
        print("="*50)
        print(f"Total attempts: {self.stats['total']}")
        print(f"✅ Success: {self.stats['success']}")
        print(f"🔍 Filtered: {self.stats['filtered']}")
        print(f"❌ Failed: {self.stats['failed']}")

        if self.stats['total'] > 0:
            success_rate = (self.stats['success'] / self.stats['total']) * 100
            print(f"📈 Overall success rate: {success_rate:.1f}%")

        # Validator stats sudah terlalu ketat, kita skip
        # self.validator.print_stats()
        self.yfinance.print_stats()

# =============================================================================
# 16. GLOBAL INDICES FETCHER (FIXED - TANDAI SEBAGAI INDEX)
# =============================================================================

class GlobalIndicesFetcher:
    def __init__(self):
        self.data = {}
        self.momentum = {}
        self.status = {}
        self.prices = {}
        self.logger = ColabLogger()

    def _to_scalar(self, value):
        if value is None:
            return 0.0
        if isinstance(value, (np.ndarray, list)):
            if len(value) > 0:
                return float(value[0])
            return 0.0
        return float(value)

    def _fetch_with_retry(self, ticker, name, attempt=1, max_retries=3):
        try:
            time.sleep(random.uniform(0.5, 1.5))

            df = yf.download(
                ticker,
                period="1y",
                interval="1d",
                auto_adjust=True,
                progress=False,
                timeout=10
            )

            if df.empty and attempt < max_retries:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                time.sleep(wait_time)
                return self._fetch_with_retry(ticker, name, attempt + 1, max_retries)

            return df

        except Exception as e:
            if attempt < max_retries:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                time.sleep(wait_time)
                return self._fetch_with_retry(ticker, name, attempt + 1, max_retries)
            return None

    def fetch_all(self, config):
        self.logger.info("Fetching global indices...")

        for name, ticker in GLOBAL_INDICES.items():
            try:
                df = self._fetch_with_retry(ticker, name)

                if df is None or df.empty or len(df) < 200:
                    self.status[name] = "UNAVAILABLE"
                    self.momentum[name] = 0.0
                    self.prices[name] = 0.0
                else:
                    # Tandai dataframe sebagai index (untuk validasi)
                    df.attrs['is_index'] = True

                    close = df['Close'].values.flatten()  # flatten untuk 1D
                    current_price = float(close[-1])

                    if len(close) >= 5:
                        momentum_array = (close[-1] / close[-5] - 1) * 100
                        momentum_value = self._to_scalar(momentum_array)
                    else:
                        momentum_value = 0.0

                    if name == "IHSG" and len(close) >= 200:
                        ma200 = np.mean(close[-200:])
                        self.data['IHSG_MA200'] = ma200
                        self.data['IHSG_Close'] = current_price

                    self.data[name] = df
                    self.momentum[name] = round(momentum_value, 2)
                    self.prices[name] = round(current_price, 2)
                    self.status[name] = "OK"

            except Exception as e:
                self.status[name] = "ERROR"
                self.momentum[name] = 0.0
                self.prices[name] = 0.0

        self.logger.success("Global indices ready")

    def get_momentum(self, name):
        return self.momentum.get(name, 0.0)

    def get_price(self, name):
        return self.prices.get(name, 0.0)

    def get_price_str(self, name):
        price = self.get_price(name)
        if price == 0:
            return "N/A"

        if name in ["IHSG", "DOWJONES"]:
            return f"{price:,.2f}"
        elif name == "USDIDR":
            return f"Rp {price:,.0f}"
        elif name == "OIL":
            return f"US$ {price:.2f}"
        elif name == "GOLD":
            return f"US$ {price:.2f}"
        return f"{price:.2f}"

    def get_trend(self, name):
        mom = self.get_momentum(name)
        if mom > 0.5:
            return "🟢 BULLISH"
        elif mom < -0.5:
            return "🔴 BEARISH"
        else:
            return "🟡 NETRAL"

    def is_ihsg_bullish(self):
        if 'IHSG_Close' in self.data and 'IHSG_MA200' in self.data:
            return self.data['IHSG_Close'] > self.data['IHSG_MA200']
        return True

# =============================================================================
# 17. KONFIGURASI DASAR (DENGAN FILTER LEBIH LONGGAR UNTUK MODAL KECIL)
# =============================================================================

class Config:
    def __init__(self, modal, mode):
        self.MODAL = modal
        self.MODE = mode

        # Adaptive filter based on modal - LEBIH LONGGAR
        if modal < 50000:
            self.MIN_PRICE = 50
            self.MIN_VOLUME = 2500  # Turunkan dari 5000
            self.MIN_AVG_VOLUME = 25000  # Turunkan dari 50000
        elif modal < 100000:
            self.MIN_PRICE = 50
            self.MIN_VOLUME = 5000
            self.MIN_AVG_VOLUME = 50000
        else:
            self.MIN_PRICE = 100
            self.MIN_VOLUME = 10000
            self.MIN_AVG_VOLUME = 100000

        self.MAX_PRICE = 50000
        self.MAX_SPREAD_PCT = 3.0  # Naikkan dari 2.0 untuk modal kecil
        self.MIN_RR = 1.0
        self.MAX_PORTFOLIO_RISK_PCT = 5.0  # Naikkan dari 3.0
        self.MIN_EV_PCT = 1.5  # Turunkan dari 2.0

        self.ENABLE_SECTOR_FILTER = True
        self.ENABLE_ENTRY_DELAY = True
        self.MAX_ENTRY_DELAY = 2
        self.ENTRY_DELAY_PROB = [0.5, 0.3, 0.2]
        self.ENABLE_RANDOM_SLIPPAGE = True
        self.ENABLE_VOLATILITY_SIZING = True

        if mode == 'intraday':
            self.INTERVAL = "1h"
            self.PERIOD = "1mo"
            self.MIN_HISTORY = 30
            self.SESSION1_START = "09:00:00"
            self.SESSION1_END = "12:00:00"
            self.SESSION2_START = "14:00:00"
            self.SESSION2_END = "16:00:00"
            self.JKT_TZ = "Asia/Jakarta"

            self.BREAKOUT_PERIOD = 10
            self.VOLUME_BREAKOUT = 1.2
            self.MA_SHORT = 20
            self.MA_LONG = 50
            self.ATR_PERIOD = 14
            self.TP_MULTIPLIER = 1.2
            self.SL_MULTIPLIER = 0.8
            self.RISK_PER_TRADE_PCT = 0.5
            self.MAX_TRADES_PER_DAY = 3
            self.FORCE_CLOSE_HOUR = 15
            self.FORCE_CLOSE_MINUTE = 50

        else:
            self.INTERVAL = "1d"
            self.PERIOD = "6mo"
            self.MIN_HISTORY = 60

            self.RSI_PERIOD = 14
            self.MA_SHORT = 20
            self.MA_LONG = 50
            self.MA200_PERIOD = 200
            self.SUPPORT_PERIOD = 60
            self.ATR_PERIOD = 14
            self.SL_MULTIPLIER = 1.5
            self.TP_MULTIPLIER = 2.5
            self.BASE_THRESHOLD = 5
            self.VOLUME_BOOST = 1.5

        # Adaptive portfolio based on modal - LEBIH FLEKSIBEL
        if modal < 50000:
            self.MAX_POSITION_PCT = 0.40  # Naikkan dari 0.30
            self.MAX_PORTFOLIO = 2
        elif modal < 100000:
            self.MAX_POSITION_PCT = 0.30
            self.MAX_PORTFOLIO = 3
        else:
            self.MAX_POSITION_PCT = 0.25
            self.MAX_PORTFOLIO = 4

class InvestasiConfig:
    def __init__(self, modal):
        self.MODAL = modal
        self.MODE = 'investasi'

        # Adaptive filter based on modal - LEBIH LONGGAR
        if modal < 50000:
            self.MIN_PRICE = 50
            self.MIN_VOLUME = 2000  # Turunkan dari 2500
            self.MIN_AVG_VOLUME = 20000  # Turunkan dari 25000
        elif modal < 100000:
            self.MIN_PRICE = 50
            self.MIN_VOLUME = 5000
            self.MIN_AVG_VOLUME = 50000
        else:
            self.MIN_PRICE = 50
            self.MIN_VOLUME = 5000
            self.MIN_AVG_VOLUME = 50000

        self.MAX_PRICE = 50000
        self.MAX_SPREAD_PCT = 8.0  # Naikkan dari 7.0
        self.MIN_EV_PCT = 0.5  # Turunkan dari 1.0

        self.QUALITY_STOCKS = QUALITY_STOCKS

        self.MA200_PERIOD = 200
        self.MA50_PERIOD = 50
        self.PULLBACK_THRESHOLD = 0.02

        self.MAX_PORTFOLIO_RISK_PCT = 15.0  # Naikkan dari 10.0
        self.MAX_PORTFOLIO = 5

        self.INTERVAL = "1d"
        self.PERIOD = "2y"
        self.MIN_HISTORY = 100

# =============================================================================
# 18. SHARED UTILITY FUNCTIONS
# =============================================================================

def normalize_columns(df):
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def calculate_return(series, period=5):
    if len(series) < period + 1:
        return 0.0
    return (series.iloc[-1] / series.iloc[-period-1] - 1) * 100

# =============================================================================
# 19. BASE STRATEGY ENGINE
# =============================================================================

class StrategyEngine:
    def __init__(self, config, global_fetcher):
        self.config = config
        self.global_fetcher = global_fetcher

    def get_signal(self, symbol, df):
        raise NotImplementedError

    def calculate_volatility_lot(self, close, atr, risk_per_trade):
        if atr <= 0:
            return None, None

        raw_lot = risk_per_trade / (atr * 100)
        lot = int(raw_lot)

        max_lot_by_modal = int(self.config.MODAL / (close * 100))
        lot = min(lot, max_lot_by_modal, 5)

        if lot >= 1:
            cost = lot * 100 * close
            if cost > self.config.MODAL:
                lot = int(self.config.MODAL / (close * 100))
                if lot >= 1:
                    cost = lot * 100 * close
                    return lot, cost
                else:
                    return None, None
            return lot, cost
        else:
            return None, None

    def calculate_fixed_lot(self, close, risk_per_lot, max_amount):
        max_lot_by_modal = int(self.config.MODAL / (close * 100))
        max_lot_by_risk = int(max_amount / (close * 100)) if max_amount >= close * 100 else 0
        max_lot = min(max_lot_by_modal, max_lot_by_risk, 5)

        if max_lot >= 1:
            lot = max_lot
            cost = lot * 100 * close
            if cost > self.config.MODAL:
                lot = int(self.config.MODAL / (close * 100))
                if lot >= 1:
                    cost = lot * 100 * close
                    return lot, cost
                else:
                    return None, None
            return lot, cost
        else:
            return None, None

# =============================================================================
# 20. ENGINE 1: SWING (SAMA PERSIS DENGAN V3.3)
# =============================================================================

class OutperformDetector:
    def __init__(self):
        self.RETURN_THRESHOLD = 2.0
        self.BEARISH_THRESHOLD = -2.0
        self.HOLD_THRESHOLD = -1.0
        self.VOLUME_THRESHOLD = 1.5

    def is_outperform(self, saham_return, ihsg_return, volume_ratio):
        if saham_return > ihsg_return + self.RETURN_THRESHOLD:
            return True, f"Return > IHSG+2%", 2
        if ihsg_return < self.BEARISH_THRESHOLD and saham_return > self.HOLD_THRESHOLD:
            return True, f"Tahan turun", 2
        if ihsg_return < self.BEARISH_THRESHOLD and volume_ratio > self.VOLUME_THRESHOLD:
            return True, f"Volume tinggi", 2
        return False, "", 0

class InflowOutflowDetector:
    def __init__(self):
        self.INFLOW_BONUS = 1
        self.OUTFLOW_PENALTY = -1

    def calculate_money_flow(self, df):
        if len(df) < 2:
            return 0, "NETRAL"
        last_close = df['Close'].iloc[-1]
        prev_close = df['Close'].iloc[-2]
        last_volume = df['Volume'].iloc[-1]
        price_change = last_close - prev_close
        money_flow = price_change * last_volume
        avg_volume = df['Volume'].tail(10).mean()
        volume_ratio = last_volume / avg_volume if avg_volume > 0 else 1
        if money_flow > 0 and volume_ratio > 1.2:
            return 1, "INFLOW"
        elif money_flow > 0:
            return 1, "INFLOW"
        elif money_flow < 0 and volume_ratio > 1.2:
            return -1, "OUTFLOW"
        elif money_flow < 0:
            return -1, "OUTFLOW"
        else:
            return 0, "NETRAL"

    def get_accumulation_distribution(self, df, period=14):
        if len(df) < period:
            return 0, "NETRAL"
        money_flow_values = []
        for i in range(-period, 0):
            if i < -1:
                price_change = df['Close'].iloc[i] - df['Close'].iloc[i-1]
                money_flow = price_change * df['Volume'].iloc[i]
                money_flow_values.append(money_flow)
        if len(money_flow_values) > 5:
            recent = sum(money_flow_values[-3:])
            previous = sum(money_flow_values[-6:-3])
            if recent > previous * 1.5:
                return 2, "AKUMULASI"
            elif recent > previous:
                return 1, "AKUMULASI"
            elif recent < previous * 0.5:
                return -2, "DISTRIBUSI"
            elif recent < previous:
                return -1, "DISTRIBUSI"
        return 0, "NETRAL"

class MultipleTouchDetector:
    def __init__(self, window=60, tolerance=1.0):
        self.window = window
        self.tolerance = tolerance / 100
        self.support_levels = {'kuat': [], 'sedang': []}
        self.resistance_levels = {'kuat': [], 'sedang': []}

    def _count_touches(self, prices, level):
        if len(prices) == 0:
            return 0
        distances = np.abs(prices - level)
        touch_threshold = level * self.tolerance
        touches = np.sum(distances < touch_threshold)
        return touches

    def _get_candidate_levels(self, low_prices, high_prices):
        all_prices = np.concatenate([low_prices, high_prices])
        if len(all_prices) == 0:
            return []
        avg_price = np.mean(all_prices)
        step = avg_price * 0.005
        min_price = np.min(all_prices) * 0.98
        max_price = np.max(all_prices) * 1.02
        candidates = np.arange(min_price, max_price + step, step)
        return candidates

    def detect_levels(self, df):
        high = df['High'].values
        low = df['Low'].values
        dates = df.index.values
        self.support_levels = {'kuat': [], 'sedang': []}
        self.resistance_levels = {'kuat': [], 'sedang': []}
        for i in range(self.window, len(high)):
            window_high = high[i-self.window:i]
            window_low = low[i-self.window:i]
            window_date = dates[i]
            candidates = self._get_candidate_levels(window_low, window_high)
            for level in candidates:
                touches_support = self._count_touches(window_low, level)
                touches_resistance = self._count_touches(window_high, level)
                if touches_support >= 3:
                    self.support_levels['kuat'].append({'price': level, 'touches': touches_support, 'date': window_date, 'strength': 'KUAT'})
                elif touches_support == 2:
                    self.support_levels['sedang'].append({'price': level, 'touches': touches_support, 'date': window_date, 'strength': 'SEDANG'})
                if touches_resistance >= 3:
                    self.resistance_levels['kuat'].append({'price': level, 'touches': touches_resistance, 'date': window_date, 'strength': 'KUAT'})
                elif touches_resistance == 2:
                    self.resistance_levels['sedang'].append({'price': level, 'touches': touches_resistance, 'date': window_date, 'strength': 'SEDANG'})
        self._deduplicate_and_sort()
        return self.support_levels, self.resistance_levels

    def _deduplicate_and_sort(self):
        def deduplicate(levels, tolerance_mult=2.0):
            if not levels:
                return []
            sorted_levels = sorted(levels, key=lambda x: x['price'])
            unique = []
            current_group = [sorted_levels[0]]
            for level in sorted_levels[1:]:
                if abs(level['price'] - current_group[-1]['price']) < (level['price'] * self.tolerance * tolerance_mult):
                    current_group.append(level)
                else:
                    best = max(current_group, key=lambda x: x['touches'])
                    unique.append(best)
                    current_group = [level]
            if current_group:
                best = max(current_group, key=lambda x: x['touches'])
                unique.append(best)
            return unique
        self.support_levels['kuat'] = deduplicate(self.support_levels['kuat'])
        self.support_levels['sedang'] = deduplicate(self.support_levels['sedang'])
        self.resistance_levels['kuat'] = deduplicate(self.resistance_levels['kuat'])
        self.resistance_levels['sedang'] = deduplicate(self.resistance_levels['sedang'])

    def get_nearest_support(self, price):
        kuat_below = [s for s in self.support_levels['kuat'] if s['price'] < price]
        if kuat_below:
            nearest = max(kuat_below, key=lambda x: x['price'])
            return nearest['price'], nearest['touches'], nearest['strength']
        sedang_below = [s for s in self.support_levels['sedang'] if s['price'] < price]
        if sedang_below:
            nearest = max(sedang_below, key=lambda x: x['price'])
            return nearest['price'], nearest['touches'], nearest['strength']
        return price * 0.95, 1, 'FALLBACK'

    def get_nearest_resistance(self, price):
        kuat_above = [r for r in self.resistance_levels['kuat'] if r['price'] > price]
        if kuat_above:
            nearest = min(kuat_above, key=lambda x: x['price'])
            return nearest['price'], nearest['touches'], nearest['strength']
        sedang_above = [r for r in self.resistance_levels['sedang'] if r['price'] > price]
        if sedang_above:
            nearest = min(sedang_above, key=lambda x: x['price'])
            return nearest['price'], nearest['touches'], nearest['strength']
        return price * 1.05, 1, 'FALLBACK'

class SwingEngine(StrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)
        self.outperform_detector = OutperformDetector()
        self.inflow_detector = InflowOutflowDetector()
        self.sr_detector = MultipleTouchDetector(window=config.SUPPORT_PERIOD, tolerance=1.0)

    def get_sector_boost(self, symbol):
        boost = 0
        ihsg_mom = self.global_fetcher.get_momentum("IHSG")
        if ihsg_mom > 0.5:
            boost += 1
        elif ihsg_mom < -0.5:
            boost -= 1
        if symbol in EXPORT_SECTOR:
            usd_mom = self.global_fetcher.get_momentum("USDIDR")
            if usd_mom > 0.5:
                boost += 1
            elif usd_mom < -0.5:
                boost -= 1
        if symbol in ENERGY_SECTOR:
            oil_mom = self.global_fetcher.get_momentum("OIL")
            if oil_mom > 1.0:
                boost += 1
            elif oil_mom < -1.0:
                boost -= 1
        if symbol in MINING_GOLD:
            gold_mom = self.global_fetcher.get_momentum("GOLD")
            if gold_mom > 1.0:
                boost += 1
            elif gold_mom < -1.0:
                boost -= 1
        return boost

    def calculate_atr_sl_tp(self, close, atr, support_price, resistance_price):
        atr = max(atr, close * 0.005)
        sl = close - (atr * self.config.SL_MULTIPLIER)
        tp_from_atr = close + (atr * self.config.TP_MULTIPLIER)
        tp_from_resistance = resistance_price * 0.98 if resistance_price > close else tp_from_atr
        tp = min(tp_from_atr, tp_from_resistance)
        sl = max(sl, close * 0.90)
        tp = min(tp, close * 1.15)
        fraction = 5 if close < 100 else 10 if close < 500 else 25
        sl = round(sl / fraction) * fraction
        tp = round(tp / fraction) * fraction
        if sl >= close or tp <= close:
            return None, None
        return int(sl), int(tp)

    def compute_features(self, df):
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']

            delta = close.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.rolling(window=self.config.RSI_PERIOD, min_periods=self.config.RSI_PERIOD).mean()
            avg_loss = loss.rolling(window=self.config.RSI_PERIOD, min_periods=self.config.RSI_PERIOD).mean()
            rs = avg_gain / (avg_loss + 1e-6)
            out['RSI'] = 100 - (100 / (1 + rs))

            out['MA20'] = close.rolling(window=self.config.MA_SHORT, min_periods=self.config.MA_SHORT).mean()
            out['MA50'] = close.rolling(window=self.config.MA_LONG, min_periods=self.config.MA_LONG).mean()
            out['MA200'] = close.rolling(window=self.config.MA200_PERIOD, min_periods=self.config.MA200_PERIOD).mean()
            out['MA_Trend'] = (out['MA20'] > out['MA50']).astype(float)
            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.config.ATR_PERIOD, min_periods=self.config.ATR_PERIOD).mean()
            out['Volume_MA'] = volume.rolling(window=20, min_periods=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)

            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out
        except Exception:
            return pd.DataFrame()

    def get_signal(self, symbol, df):
        try:
            self.sr_detector.detect_levels(df)
            df_feat = self.compute_features(df)
            if len(df_feat) < self.config.MIN_HISTORY:
                return None

            latest = df_feat.iloc[-1]
            close = float(latest['Close'])

            ma200 = float(latest['MA200']) if not pd.isna(latest['MA200']) else close
            if close < ma200:
                return None

            prev_close = float(df['Close'].iloc[-2]) if len(df) >= 2 else close
            if close <= prev_close:
                return None

            saham_return = calculate_return(df['Close'], 5)
            ihsg_return = self.global_fetcher.get_momentum("IHSG")
            volume_ratio = float(latest['Volume_Ratio']) if not pd.isna(latest['Volume_Ratio']) else 1

            is_outperform, outperform_reason, outperform_bonus = self.outperform_detector.is_outperform(
                saham_return, ihsg_return, volume_ratio
            )

            inflow_score, inflow_trend = self.inflow_detector.calculate_money_flow(df)
            acc_score, acc_trend = self.inflow_detector.get_accumulation_distribution(df)

            support_price, support_touches, support_strength = self.sr_detector.get_nearest_support(close)
            resistance_price, resistance_touches, resistance_strength = self.sr_detector.get_nearest_resistance(close)
            dist_to_support = (close - support_price) / close * 100

            rsi = float(latest['RSI']) if not pd.isna(latest['RSI']) else 50
            atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.02
            ma20 = float(latest['MA20']) if not pd.isna(latest['MA20']) else close
            ma50 = float(latest['MA50']) if not pd.isna(latest['MA50']) else close
            ma_trend = float(latest['MA_Trend']) if not pd.isna(latest['MA_Trend']) else 0

            base_score = 0
            if rsi < 30:
                base_score += 3
            elif rsi < 40:
                base_score += 1
            if volume_ratio > self.config.VOLUME_BOOST:
                base_score += 2
            elif volume_ratio > 1.2:
                base_score += 1
            if dist_to_support < 3:
                if support_strength == 'KUAT':
                    base_score += 2
                elif support_strength == 'SEDANG':
                    base_score += 1
                else:
                    base_score += 1
            if ma_trend > 0.5:
                base_score += 1

            global_boost = self.get_sector_boost(symbol)
            inflow_bonus = 1 if inflow_score > 0 else -1 if inflow_score < 0 else 0
            accumulation_bonus = 1 if acc_score > 0 else -1 if acc_score < 0 else 0
            score = base_score + global_boost + outperform_bonus + inflow_bonus + accumulation_bonus

            ihsg_momentum = self.global_fetcher.get_momentum("IHSG")
            if ihsg_momentum < -2.0:
                effective_threshold = self.config.BASE_THRESHOLD + 1
                position_multiplier = 0.5
            elif ihsg_momentum < -1.0:
                effective_threshold = self.config.BASE_THRESHOLD
                position_multiplier = 0.75
            else:
                effective_threshold = self.config.BASE_THRESHOLD
                position_multiplier = 1.0

            sl, tp = self.calculate_atr_sl_tp(close, atr, support_price, resistance_price)
            if sl is None or tp is None:
                return None

            risk = close - sl
            reward = tp - close
            if risk <= 0 or reward <= 0:
                return None

            rr = reward / risk
            if rr < self.config.MIN_RR:
                return None

            prob_up = 0.5 + (score * 0.03)
            prob_up = min(max(prob_up, 0.3), 0.8)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close) * 100

            if ev_pct < self.config.MIN_EV_PCT:
                return None

            risk_per_trade = self.config.MODAL * (self.config.RISK_PER_TRADE_PCT / 100) * position_multiplier
            lot, cost = self.calculate_volatility_lot(close, atr, risk_per_trade)

            if lot is None or cost is None:
                return None

            sector = get_sector(symbol)

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close),
                'RSI': round(rsi, 1),
                'Support': int(support_price),
                'Resistance': int(resistance_price),
                'Stop_Loss': sl,
                'Take_Profit': tp,
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV': int(expected_value),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'Risk': int(risk),
                'ATR': round(atr, 2),
                'Lot': lot,
                'Cost': cost,
                'Inflow': inflow_trend,
                'Acc': acc_trend,
                'Volume': f"{volume_ratio:.1f}x",
                'Reasons': f"RSI {rsi:.0f}, Vol {volume_ratio:.1f}x, {support_strength}",
                'Chart': "BULLISH" if ma20 > ma50 and rsi > 50 else "BEARISH" if ma20 < ma50 and rsi < 50 else "NETRAL"
            }
        except Exception as e:
            return None

# =============================================================================
# 21. ENGINE 2: INTRADAY LIQUID
# =============================================================================

class IntradayLiquidEngine(StrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)

    def compute_features(self, df):
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']

            out['Highest_High'] = high.shift(1).rolling(window=self.config.BREAKOUT_PERIOD).max()
            out['MA_Short'] = close.rolling(window=self.config.MA_SHORT).mean()
            out['MA_Long'] = close.rolling(window=self.config.MA_LONG).mean()
            out['MA_Alignment'] = (out['MA_Short'] > out['MA_Long']).astype(int)
            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)
            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.config.ATR_PERIOD).mean()
            out['ATR_Pct'] = out['ATR'] / (close + 1e-6) * 100
            out['Body'] = abs(close - out['Open'])
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)
            out['Upper_Wick'] = (high - out[['Close', 'Open']].max(axis=1)) / (out['Range'] + 1e-6)
            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out
        except Exception:
            return pd.DataFrame()

    def check_breakout(self, row):
        if pd.isna(row['Highest_High']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High']:
            return False
        if row['Volume_Ratio'] < self.config.VOLUME_BREAKOUT:
            return False
        if row['MA_Alignment'] != 1:
            return False
        return True

    def get_signal(self, symbol, df):
        try:
            df_feat = self.compute_features(df)
            if len(df_feat) < 30:
                return None
            latest = df_feat.iloc[-1]
            close = float(latest['Close'])
            if not self.check_breakout(latest):
                return None
            atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.02
            atr = max(atr, close * 0.005)
            sl = close - (atr * self.config.SL_MULTIPLIER)
            tp = close + (atr * self.config.TP_MULTIPLIER)
            fraction = 5 if close < 100 else 10 if close < 500 else 25
            sl = round(sl / fraction) * fraction
            tp = round(tp / fraction) * fraction
            if sl >= close or tp <= close:
                return None
            risk = close - sl
            reward = tp - close
            rr = reward / risk
            if rr < self.config.MIN_RR:
                return None
            score = 5
            if latest['Volume_Ratio'] > 1.5:
                score += 1
            if latest['Body_Ratio'] > 0.6:
                score += 1
            if latest['Upper_Wick'] < 0.3:
                score += 1
            prob_up = 0.5 + (score * 0.02)
            prob_up = min(prob_up, 0.7)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close) * 100
            if ev_pct < self.config.MIN_EV_PCT:
                return None

            risk_per_trade = self.config.MODAL * (self.config.RISK_PER_TRADE_PCT / 100)
            lot, cost = self.calculate_volatility_lot(close, atr, risk_per_trade)

            if lot is None or cost is None:
                return None

            flow = "INFLOW" if latest['Volume_Ratio'] > 1.2 else "NETRAL"
            acc = "AKUMULASI" if latest['Volume_Ratio'] > 1.3 else "NETRAL"
            sector = get_sector(symbol)

            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close),
                'RSI': '-',
                'Support': int(latest['Highest_High']),
                'Resistance': '-',
                'Stop_Loss': int(sl),
                'Take_Profit': int(tp),
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV': int(expected_value),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'Risk': int(risk),
                'ATR': round(atr, 2),
                'Lot': lot,
                'Cost': cost,
                'Inflow': flow,
                'Acc': acc,
                'Volume': f"{latest['Volume_Ratio']:.1f}x",
                'Body_Ratio': f"{latest['Body_Ratio']:.2f}",
                'Upper_Wick': f"{latest['Upper_Wick']:.2f}",
                'Reasons': f"Breakout {self.config.BREAKOUT_PERIOD}, Vol {latest['Volume_Ratio']:.1f}x",
                'Chart': "BULLISH" if latest['MA_Alignment'] == 1 else "NETRAL"
            }
        except Exception:
            return None

# =============================================================================
# 22. ENGINE 3: INTRADAY GORENGAN
# =============================================================================

class IntradayGorenganEngine(StrategyEngine):
    def __init__(self, config, global_fetcher):
        super().__init__(config, global_fetcher)

    def compute_features(self, df):
        try:
            out = df.copy()
            close = out['Close']
            high = out['High']
            low = out['Low']
            volume = out['Volume']
            open_price = out['Open']

            out['Highest_High_5'] = high.shift(1).rolling(window=5).max()
            out['Volume_MA'] = volume.rolling(window=20).mean()
            out['Volume_Ratio'] = volume / (out['Volume_MA'] + 1e-6)
            out['Turnover'] = close * volume
            out['TR'] = np.maximum(high - low, np.maximum((high - close.shift()).abs(), (low - close.shift()).abs()))
            out['ATR'] = out['TR'].rolling(window=self.config.ATR_PERIOD).mean()
            out['Body'] = abs(close - open_price)
            out['Range'] = high - low
            out['Body_Ratio'] = out['Body'] / (out['Range'] + 1e-6)
            out['Day_Change'] = (close / open_price - 1) * 100
            out = out.replace([np.inf, -np.inf], np.nan)
            out = out.dropna()
            return out
        except Exception:
            return pd.DataFrame()

    def check_breakout(self, row):
        if pd.isna(row['Highest_High_5']) or pd.isna(row['Close']):
            return False
        if row['Close'] <= row['Highest_High_5']:
            return False
        if row['Volume_Ratio'] < 2.0:
            return False
        min_turnover = self.config.MODAL * 10
        if row['Turnover'] < min_turnover:
            return False
        if row['Day_Change'] > 20:
            return False
        return True

    def get_signal(self, symbol, df):
        try:
            df_feat = self.compute_features(df)
            if len(df_feat) < 30:
                return None
            latest = df_feat.iloc[-1]
            close = float(latest['Close'])
            if not self.check_breakout(latest):
                return None
            atr = float(latest['ATR']) if not pd.isna(latest['ATR']) else close * 0.03
            atr = max(atr, close * 0.01)
            sl = close - (atr * 1.0)
            tp = close + (atr * 1.5)
            fraction = 5 if close < 100 else 10 if close < 500 else 25
            sl = round(sl / fraction) * fraction
            tp = round(tp / fraction) * fraction
            if sl >= close or tp <= close:
                return None
            risk = close - sl
            reward = tp - close
            rr = reward / risk
            if rr < self.config.MIN_RR:
                return None
            score = 5
            if latest['Volume_Ratio'] > 3:
                score += 2
            elif latest['Volume_Ratio'] > 2:
                score += 1
            if latest['Body_Ratio'] > 0.7:
                score += 1
            prob_up = 0.5 + (score * 0.02)
            prob_up = min(prob_up, 0.7)
            expected_value = (prob_up * reward) - ((1 - prob_up) * risk)
            ev_pct = (expected_value / close) * 100
            if ev_pct < 1.5:
                return None
            risk_per_trade = self.config.MODAL * 0.003
            lot, cost = self.calculate_volatility_lot(close, atr, risk_per_trade)
            if lot is None or cost is None:
                return None
            sector = get_sector(symbol)
            return {
                'Symbol': symbol,
                'Sector': sector,
                'Price': int(close),
                'RSI': '-',
                'Support': int(latest['Highest_High_5']),
                'Resistance': '-',
                'Stop_Loss': int(sl),
                'Take_Profit': int(tp),
                'R/R': round(rr, 2),
                'Prob_Up': round(prob_up, 3),
                'EV': int(expected_value),
                'EV_Pct': round(ev_pct, 2),
                'Score': score,
                'Risk': int(risk),
                'ATR': round(atr, 2),
                'Volume_Spike': f"{latest['Volume_Ratio']:.1f}x",
                'Turnover': f"Rp {latest['Turnover']/1e6:.0f}Jt",
                'Lot': lot,
                'Cost': cost,
                'Inflow': 'INFLOW' if latest['Volume_Ratio'] > 1.5 else 'NETRAL',
                'Acc': 'AKUMULASI' if latest['Volume_Ratio'] > 2 else 'NETRAL',
                'Volume': f"{latest['Volume_Ratio']:.1f}x",
                'Reasons': f"Breakout 5, Vol {latest['Volume_Ratio']:.1f}x",
                'Chart': "BULLISH" if latest['Day_Change'] > 0 else "NETRAL"
            }
        except Exception:
            return None

# =============================================================================
# 23. ENGINE 4: INVESTASI
# =============================================================================

class InvestasiEngine:
    def __init__(self, config, global_fetcher):
        self.config = config
        self.global_fetcher = global_fetcher
        self.quality_stocks = config.QUALITY_STOCKS
        self.quality_estimator = QualityEstimator()
        self.rti_scraper = RTIScraper()

        self.MA200_STRICT = True
        self.MA50_LOWER = -20
        self.MA50_UPPER = 15

    def get_signal(self, symbol, df):
        if symbol not in self.quality_stocks:
            return None

        if len(df) < self.config.MIN_HISTORY:
            return None

        close = df['Close']

        if len(df) >= 200:
            ma200 = close.rolling(window=200).mean()
            current_ma200 = float(ma200.iloc[-1]) if not pd.isna(ma200.iloc[-1]) else None
        else:
            ma100 = close.rolling(window=100).mean()
            current_ma200 = float(ma100.iloc[-1]) if not pd.isna(ma100.iloc[-1]) else close.iloc[-1] * 0.9

        if len(df) >= 50:
            ma50 = close.rolling(window=50).mean()
            current_ma50 = float(ma50.iloc[-1]) if not pd.isna(ma50.iloc[-1]) else close.iloc[-1] * 0.95
        else:
            ma30 = close.rolling(window=30).mean()
            current_ma50 = float(ma30.iloc[-1]) if not pd.isna(ma30.iloc[-1]) else close.iloc[-1] * 0.95

        current_price = float(close.iloc[-1])

        if current_ma200 is not None and current_price < current_ma200:
            return None

        ma200_bonus = 1 if current_ma200 is not None and current_price > current_ma200 else 0

        price_to_ma50 = (current_price / current_ma50 - 1) * 100 if current_ma50 else 0

        if price_to_ma50 > self.MA50_UPPER:
            return None

        if price_to_ma50 < self.MA50_LOWER:
            return None

        quality = self.quality_estimator.estimate(symbol, df)
        fundamental = self.rti_scraper.scrape(symbol)

        max_amount = self.config.MODAL * 0.3
        lot = int(max_amount / (current_price * 100))
        lot = max(1, min(lot, 5))
        cost = lot * 100 * current_price

        if cost > self.config.MODAL:
            lot = int(self.config.MODAL / (current_price * 100))
            if lot < 1:
                return None
            cost = lot * 100 * current_price

        stop_loss = current_price * 0.80
        sector = get_sector(symbol)

        final_score = quality['score'] + (ma200_bonus * 10)

        return {
            'Symbol': symbol,
            'Sector': sector,
            'Price': int(current_price),
            'RSI': '-',
            'MA50': int(current_ma50) if current_ma50 else '-',
            'MA200': int(current_ma200) if current_ma200 else '-',
            'To_MA50': f"{price_to_ma50:.1f}%",
            'PER': fundamental.get('per', 'N/A'),
            'PBV': fundamental.get('pbv', 'N/A'),
            'MarketCap': fundamental.get('market_cap', 'N/A'),
            'Stop_Loss': int(stop_loss),
            'Take_Profit': 'HOLD',
            'Lot': lot,
            'Cost': cost,
            'Risk': int(current_price - stop_loss),
            'Risk%': round((current_price - stop_loss) / current_price * 100, 1),
            'Quality': quality['category'],
            'Quality_Score': quality['score'],
            'MA200_Bonus': ma200_bonus,
            'Final_Score': final_score,
            'Reasons': f'Pullback ke MA50 ({price_to_ma50:.1f}%), {quality["category"]}',
            'Chart': 'BULLISH' if current_price > (current_ma200 or current_price) else 'NEUTRAL',
            'Inflow': 'QUALITY',
            'Acc': 'LONG-TERM'
        }

# =============================================================================
# 24. BACKTESTER (SINGKAT)
# =============================================================================

class MonteCarloSimulator:
    def __init__(self, trades, initial_capital=40000, n_simulations=1000):
        self.trades = trades
        self.initial_capital = initial_capital
        self.n_simulations = n_simulations
        self.results = []
        self.summary = {}

    def run(self):
        if not self.trades:
            print("   ❌ No trades to simulate")
            return

        print(f"\n🎲 Running Monte Carlo ({self.n_simulations} simulations)...")

        returns = [t['return_after_fee_pct'] / 100 for t in self.trades]

        for sim in range(self.n_simulations):
            sampled_returns = random.choices(returns, k=len(returns))
            equity = self.initial_capital
            peak = equity
            max_dd = 0
            final_equity = equity

            for r in sampled_returns:
                equity = equity * (1 + r)
                if equity > peak:
                    peak = equity
                else:
                    dd = (peak - equity) / peak * 100
                    if dd > max_dd:
                        max_dd = dd
                final_equity = equity

            total_return_pct = (final_equity - self.initial_capital) / self.initial_capital * 100
            self.results.append(total_return_pct)

        returns_pct = self.results

        self.summary = {
            'n_simulations': self.n_simulations,
            'mean_return': np.mean(returns_pct),
            'median_return': np.median(returns_pct),
            'std_return': np.std(returns_pct),
            'percentile_5': np.percentile(returns_pct, 5),
            'percentile_95': np.percentile(returns_pct, 95),
            'min_return': min(returns_pct),
            'max_return': max(returns_pct),
            'pct_positive': np.sum(np.array(returns_pct) > 0) / self.n_simulations * 100
        }

        print(f"   ✅ Monte Carlo complete")
        return self.summary

    def print_results(self):
        if not self.summary:
            print("\n📊 No Monte Carlo results")
            return

        print("\n" + "="*100)
        print("🎲 MONTE CARLO SIMULATION RESULTS")
        print("="*100)
        print(f"Number of simulations: {self.summary['n_simulations']:,}")
        print(f"Based on: {len(self.trades)} historical trades")
        print(f"Initial capital: Rp {self.initial_capital:,.0f}")

        print("\n📈 RETURN DISTRIBUTION:")
        print(f"   Mean return: {self.summary['mean_return']:.2f}%")
        print(f"   Median return: {self.summary['median_return']:.2f}%")
        print(f"   Std deviation: {self.summary['std_return']:.2f}%")
        print(f"   Best case (95th percentile): {self.summary['percentile_95']:.2f}%")
        print(f"   Worst case (5th percentile): {self.summary['percentile_5']:.2f}%")
        print(f"   Range: {self.summary['min_return']:.2f}% to {self.summary['max_return']:.2f}%")
        print(f"   Probability of profit: {self.summary['pct_positive']:.1f}%")

class Backtester:
    def __init__(self, config, global_fetcher, engine):
        self.config = config
        self.global_fetcher = global_fetcher
        self.engine = engine
        self.trades = []
        self.metrics = {
            'total_trades': 0, 'winning_trades': 0, 'losing_trades': 0,
            'total_return': 0, 'returns': [], 'returns_after_fee': []
        }
        self.equity_curve = []
        self.equity_curve_after_fee = []
        self.max_drawdown = 0
        self.max_drawdown_after_fee = 0
        self.max_drawdown_pct = 0
        self.max_drawdown_pct_after_fee = 0
        self.total_fees = 0
        self.monte_carlo = None
        self.entry_delay_stats = {'delay_0': 0, 'delay_1': 0, 'delay_2': 0}

    def calculate_equity_curve(self, initial_capital=100000):
        if not self.trades:
            return [], [], 0, 0
        sorted_trades = sorted(self.trades, key=lambda x: x['entry_date'])
        equity = initial_capital
        curve = [(sorted_trades[0]['entry_date'], initial_capital)]
        peak = initial_capital
        max_dd = 0
        equity_fee = initial_capital
        curve_fee = [(sorted_trades[0]['entry_date'], initial_capital)]
        peak_fee = initial_capital
        max_dd_fee = 0
        total_fees = 0
        for trade in sorted_trades:
            equity = equity * (1 + trade['return_pct'] / 100)
            curve.append((trade['entry_date'], equity))
            if equity > peak:
                peak = equity
            else:
                dd = (peak - equity) / peak * 100
                if dd > max_dd:
                    max_dd = dd
            return_after_fee = trade.get('return_after_fee_pct', trade['return_pct'])
            equity_fee = equity_fee * (1 + return_after_fee / 100)
            curve_fee.append((trade['entry_date'], equity_fee))
            if equity_fee > peak_fee:
                peak_fee = equity_fee
            else:
                dd_fee = (peak_fee - equity_fee) / peak_fee * 100
                if dd_fee > max_dd_fee:
                    max_dd_fee = dd_fee
            total_fees += trade.get('fee_cost', 0)
        self.equity_curve = curve
        self.equity_curve_after_fee = curve_fee
        self.max_drawdown_pct = max_dd
        self.max_drawdown_pct_after_fee = max_dd_fee
        self.total_fees = total_fees
        return curve, curve_fee, max_dd, max_dd_fee

    def run_monte_carlo(self, n_simulations=1000):
        if len(self.trades) < 20:
            print("\n⚠️ Monte Carlo: Minimal 20 trades required")
            return None

        self.monte_carlo = MonteCarloSimulator(
            trades=self.trades,
            initial_capital=self.config.MODAL,
            n_simulations=n_simulations
        )
        return self.monte_carlo.run()

    def get_entry_price_with_delay(self, df, signal_idx, signal_price):
        if not hasattr(self.config, 'ENABLE_ENTRY_DELAY') or not self.config.ENABLE_ENTRY_DELAY:
            return signal_price, 0

        delay = random.choices(
            [0, 1, 2],
            weights=self.config.ENTRY_DELAY_PROB
        )[0]

        self.entry_delay_stats[f'delay_{delay}'] += 1

        if delay == 0:
            return signal_price, delay

        max_idx = len(df) - 1
        entry_idx = min(signal_idx + delay, max_idx)

        if entry_idx == signal_idx:
            return signal_price, 0

        next_close = float(df.iloc[entry_idx]['Close'])
        entry_price = max(next_close, signal_price)

        return entry_price, delay

    def run(self, stocks_data, time_step=1):
        print(f"\n📊 Running backtest...")
        print(f"   📈 Saham: {len(stocks_data)}")
        print(f"   ⏱️  Time step: setiap {time_step} hari")

        total_signals = 0
        total_trades = 0
        winning_trades = 0
        losing_trades = 0
        all_returns = []
        all_returns_after_fee = []

        for symbol, df in stocks_data.items():
            if len(df) < 60:
                continue

            for i in range(60, len(df) - 5, time_step):
                try:
                    data_hingga_i = df.iloc[:i].copy()
                    signal = self.engine.get_signal(symbol, data_hingga_i)
                    if signal:
                        total_signals += 1

                        signal_price = signal['Price']
                        entry_price, delay_used = self.get_entry_price_with_delay(
                            df, i, signal_price
                        )

                        sl = signal['Stop_Loss']
                        tp = signal['Take_Profit']
                        lot = signal['Lot']

                        data_setelah = df.iloc[i + delay_used : i + delay_used + 5]

                        if len(data_setelah) > 0:
                            hit_sl = False
                            hit_tp = False
                            exit_price = entry_price

                            for j in range(len(data_setelah)):
                                high = data_setelah.iloc[j]['High']
                                low = data_setelah.iloc[j]['Low']

                                if low <= sl and high >= tp:
                                    if random.random() < 0.5:
                                        hit_sl = True
                                        exit_price = sl
                                    else:
                                        hit_tp = True
                                        exit_price = tp
                                    break
                                elif low <= sl:
                                    hit_sl = True
                                    exit_price = sl
                                    break
                                elif high >= tp:
                                    hit_tp = True
                                    exit_price = tp
                                    break

                            if not hit_sl and not hit_tp:
                                exit_price = data_setelah.iloc[-1]['Close']

                            return_pct = (exit_price - entry_price) / entry_price * 100
                            return_after_fee, fee_cost = FeeConfig.adjust_return_for_fee(entry_price, exit_price, lot)

                            self.trades.append({
                                'symbol': symbol,
                                'entry_date': df.index[i + delay_used],
                                'signal_date': df.index[i],
                                'delay': delay_used,
                                'signal_price': signal_price,
                                'entry_price': entry_price,
                                'exit_price': exit_price,
                                'return_pct': round(return_pct, 2),
                                'return_after_fee_pct': round(return_after_fee, 2),
                                'fee_cost': round(fee_cost, 0),
                                'lot': lot,
                                'hit_sl': hit_sl,
                                'hit_tp': hit_tp
                            })
                            all_returns.append(return_pct)
                            all_returns_after_fee.append(return_after_fee)
                            total_trades += 1
                            if return_after_fee > 0:
                                winning_trades += 1
                            else:
                                losing_trades += 1
                except Exception:
                    continue

        self.metrics = {
            'total_signals': total_signals,
            'total_trades': total_trades,
            'winning_trades': winning_trades,
            'losing_trades': losing_trades,
            'total_return': sum(all_returns) if all_returns else 0,
            'total_return_after_fee': sum(all_returns_after_fee) if all_returns_after_fee else 0,
            'avg_return': np.mean(all_returns) if all_returns else 0,
            'avg_return_after_fee': np.mean(all_returns_after_fee) if all_returns_after_fee else 0,
            'returns': all_returns,
            'returns_after_fee': all_returns_after_fee
        }

        if total_trades > 0:
            self.calculate_equity_curve(initial_capital=self.config.MODAL)

        print(f"   ✅ Backtest complete: {total_trades} trades")
        print(f"   💰 Total fees: Rp {self.total_fees:,.0f}")
        return self.metrics

    def print_results(self):
        if self.metrics['total_trades'] == 0:
            print("\n📊 No backtest results")
            return

        print("\n" + "="*100)
        print("📊 BACKTEST RESULTS (Dengan Fee Realism)")
        print("="*100)

        total = self.metrics['total_trades']
        win = self.metrics['winning_trades']
        loss = self.metrics['losing_trades']
        win_rate = (win / total * 100) if total > 0 else 0

        print(f"Total Trades: {total}")
        print(f"Winning Trades: {win}")
        print(f"Losing Trades: {loss}")
        print(f"Win Rate: {win_rate:.1f}%")

        if total > 0:
            avg_return = self.metrics['avg_return']
            avg_return_fee = self.metrics['avg_return_after_fee']
            total_return = self.metrics['total_return']
            total_return_fee = self.metrics['total_return_after_fee']

            print(f"\n📈 SEBELUM FEE:")
            print(f"   Average Return: {avg_return:.2f}%")
            print(f"   Total Return (sum): {total_return:.2f}%")

            print(f"\n📉 SETELAH FEE (dengan biaya riil):")
            print(f"   Average Return: {avg_return_fee:.2f}%")
            print(f"   Total Return (sum): {total_return_fee:.2f}%")
            print(f"   Total Biaya Fee: Rp {self.total_fees:,.0f}")
            print(f"   Fee sebagai % dari modal: {(self.total_fees/self.config.MODAL)*100:.2f}%")

            if loss > 0:
                avg_win = np.mean([r for r in self.metrics['returns'] if r > 0]) if win > 0 else 0
                avg_loss = abs(np.mean([r for r in self.metrics['returns'] if r < 0])) if loss > 0 else 0
                profit_factor = (avg_win * win) / (avg_loss * loss) if avg_loss > 0 else float('inf')
                avg_win_fee = np.mean([r for r in self.metrics['returns_after_fee'] if r > 0]) if win > 0 else 0
                avg_loss_fee = abs(np.mean([r for r in self.metrics['returns_after_fee'] if r < 0])) if loss > 0 else 0
                profit_factor_fee = (avg_win_fee * win) / (avg_loss_fee * loss) if avg_loss_fee > 0 else float('inf')
                print(f"\n📊 PROFIT FACTOR:")
                print(f"   Sebelum Fee: {profit_factor:.2f}")
                print(f"   Setelah Fee: {profit_factor_fee:.2f}")

        if self.equity_curve:
            start_equity = self.equity_curve[0][1]
            end_equity = self.equity_curve[-1][1]
            end_equity_fee = self.equity_curve_after_fee[-1][1]
            total_return_pct = (end_equity - start_equity) / start_equity * 100
            total_return_pct_fee = (end_equity_fee - start_equity) / start_equity * 100

            print("\n" + "-"*50)
            print("📈 EQUITY CURVE (Compounding)")
            print("-"*50)
            print(f"Start Equity: Rp {start_equity:,.0f}")
            print(f"End Equity (sebelum fee): Rp {end_equity:,.0f} ({total_return_pct:.2f}%)")
            print(f"End Equity (setelah fee): Rp {end_equity_fee:,.0f} ({total_return_pct_fee:.2f}%)")
            print(f"Max Drawdown (sebelum fee): {self.max_drawdown_pct:.2f}%")
            print(f"Max Drawdown (setelah fee): {self.max_drawdown_pct_after_fee:.2f}%")
            print(f"Number of Trades: {len(self.equity_curve)-1}")

# =============================================================================
# 25. STOCK SCANNER
# =============================================================================

class StockScanner:
    def __init__(self, config, global_fetcher, engine, state_manager=None):
        self.config = config
        self.global_fetcher = global_fetcher
        self.engine = engine
        self.daily_trade_count = 0
        self.state_manager = state_manager
        self.logger = ColabLogger()
        self.quality_estimator = QualityEstimator()
        self.market_regime = MarketRegimeDetector(global_fetcher)
        self.rti_scraper = RTIScraper()

    def calculate_portfolio_risk(self, selected_signals):
        total_risk = 0
        for signal in selected_signals:
            if 'Risk' in signal:
                risk_per_lot = signal['Risk'] * 100
                total_risk += risk_per_lot * signal['Lot']
        risk_pct = (total_risk / self.config.MODAL) * 100 if self.config.MODAL > 0 else 0
        return total_risk, risk_pct

    def filter_by_ranking(self, signals):
        if not signals:
            return []

        if hasattr(self.config, 'MIN_EV_PCT'):
            ev_filtered = [s for s in signals if s.get('EV_Pct', 100) >= self.config.MIN_EV_PCT]
        else:
            ev_filtered = signals

        if not ev_filtered:
            return []

        if 'Final_Score' in ev_filtered[0]:
            ranked = sorted(ev_filtered, key=lambda x: -x['Final_Score'])
        elif 'Score' in ev_filtered[0]:
            ranked = sorted(ev_filtered, key=lambda x: -x['Score'])
        else:
            ranked = ev_filtered

        max_positions = getattr(self.config, 'MAX_PORTFOLIO', 5)
        risk_cap = getattr(self.config, 'MAX_PORTFOLIO_RISK_PCT', 100)

        selected = []
        total_risk_pct = 0

        for signal in ranked:
            risk_amount = signal.get('Risk', 0) * signal.get('Lot', 1) * 100
            risk_pct = (risk_amount / self.config.MODAL) * 100 if self.config.MODAL > 0 else 0

            if len(selected) < max_positions and total_risk_pct + risk_pct <= risk_cap:
                selected.append(signal)
                total_risk_pct += risk_pct
            else:
                break

        return selected

    def compare_with_previous(self, current_signals):
        if not self.state_manager:
            return None

        previous_state = self.state_manager.load_latest_state()
        if not previous_state:
            return None

        prev_data = previous_state.get('data', {})
        prev_signals = prev_data.get('signals', [])

        if not prev_signals:
            return None

        prev_symbols = {s['Symbol'] for s in prev_signals}
        curr_symbols = {s['Symbol'] for s in current_signals}

        persistent = {}
        for symbol in curr_symbols & prev_symbols:
            days = 1
            history = self.state_manager.load_history(days=7)
            for state in history[:3]:
                hist_data = state.get('data', {})
                hist_signals = hist_data.get('signals', [])
                hist_symbols = {s['Symbol'] for s in hist_signals}
                if symbol in hist_symbols:
                    days += 1
                else:
                    break
            persistent[symbol] = days

        return {
            'new': curr_symbols - prev_symbols,
            'lost': prev_symbols - curr_symbols,
            'persistent': persistent,
            'total_prev': len(prev_signals),
            'total_curr': len(current_signals)
        }

    def print_global_summary(self):
        print("\n" + "="*100)
        print("🌍 RINGKASAN PASAR")
        print("="*100)
        data = []
        for name in GLOBAL_INDICES.keys():
            mom = self.global_fetcher.get_momentum(name)
            trend = self.global_fetcher.get_trend(name)
            price_str = self.global_fetcher.get_price_str(name)
            data.append([name, price_str, f"{mom:+.2f}%", trend])
        print(tabulate(data, headers=["Indeks", "Harga", "Momentum", "Trend"], tablefmt="grid"))

    def print_comparison(self, comparison):
        if not comparison:
            return

        print("\n" + "="*100)
        print("📊 PERBANDINGAN DENGAN SCAN KEMARIN")
        print("="*100)

        print(f"Total sinyal kemarin: {comparison['total_prev']}")
        print(f"Total sinyal hari ini: {comparison['total_curr']}")
        print(f"Perubahan: {comparison['total_curr'] - comparison['total_prev']:+d}")

        if comparison['new']:
            print(f"\n🟢 SAHAM BARU ({len(comparison['new'])}):")
            for symbol in sorted(list(comparison['new']))[:10]:
                print(f"   - {symbol}")
            if len(comparison['new']) > 10:
                print(f"   ... dan {len(comparison['new'])-10} lainnya")

        if comparison['lost']:
            print(f"\n🔴 SAHAM HILANG ({len(comparison['lost'])}):")
            for symbol in sorted(list(comparison['lost']))[:10]:
                print(f"   - {symbol}")
            if len(comparison['lost']) > 10:
                print(f"   ... dan {len(comparison['lost'])-10} lainnya")

        if comparison['persistent']:
            print(f"\n🟡 SAHAM BERTAHAN:")
            persistent_sorted = sorted(comparison['persistent'].items(), key=lambda x: -x[1])
            for symbol, days in persistent_sorted[:10]:
                print(f"   - {symbol}: {days} hari berturut-turut")

    def print_signals(self, signals, engine_name):
        if not signals:
            print(f"\n❌ Tidak ada sinyal {engine_name} hari ini")
            return

        print("\n" + "="*100)
        print(f"📊 {engine_name} - REKOMENDASI ({len(signals)} sinyal)")
        print("="*100)
        print(f"Modal: Rp {self.config.MODAL:,}")

        if hasattr(self.config, 'MAX_PORTFOLIO'):
            print(f"Max posisi: {self.config.MAX_PORTFOLIO}")
        if hasattr(self.config, 'MAX_PORTFOLIO_RISK_PCT'):
            print(f"Portfolio Risk Cap: {self.config.MAX_PORTFOLIO_RISK_PCT}%")

        print("-"*100)

        display_data = []
        for s in signals:
            risk_amount = s.get('Risk', 0) * s.get('Lot', 1) * 100
            risk_pct = (risk_amount / self.config.MODAL) * 100 if self.config.MODAL > 0 else 0

            if engine_name == "INVESTASI ENGINE":
                display_data.append([
                    s['Symbol'],
                    s['Sector'],
                    f"Rp {s['Price']:,}",
                    f"{s['To_MA50']}",
                    f"Rp {s.get('MA50', '-')}",
                    f"Rp {s.get('MA200', '-')}",
                    s.get('PER', 'N/A'),
                    s.get('PBV', 'N/A'),
                    f"Rp {s['Stop_Loss']:,}",
                    s['Take_Profit'],
                    f"{risk_pct:.1f}%",
                    f"{s['Lot']} lot",
                    f"Rp {s['Cost']:,}",
                    s['Quality'],
                    "⭐" if s['MA200_Bonus'] else "-"
                ])
                headers = [
                    "Kode", "Sektor", "Harga", "To MA50", "MA50", "MA200",
                    "PER", "PBV", "Stop Loss", "Target", "Risk%", "Lot", "Biaya", "Quality", "MA200+"
                ]
            else:
                flow_status = "💰 INFLOW" if s.get('Inflow') == "INFLOW" else "💸 OUTFLOW" if s.get('Inflow') == "OUTFLOW" else "⚖️ NETRAL"
                acc_status = "📈 AKUM" if s.get('Acc') == "AKUMULASI" else "📉 DIST" if s.get('Acc') == "DISTRIBUSI" else "➖"

                display_data.append([
                    s['Symbol'],
                    s['Sector'],
                    f"Rp {s['Price']:,}",
                    s.get('RSI', '-'),
                    f"{s.get('Volume', '-')}",
                    flow_status,
                    acc_status,
                    f"Rp {s.get('Support', 0):,}",
                    f"Rp {s.get('Resistance', 0):,}" if s.get('Resistance') else '-',
                    f"Rp {s.get('Stop_Loss', 0):,}",
                    f"Rp {s.get('Take_Profit', 0):,}",
                    f"1:{s.get('R/R', 0)}",
                    f"{s.get('Prob_Up', 0):.0%}" if s.get('Prob_Up') else '-',
                    f"{s.get('EV_Pct', 0)}%",
                    f"{s.get('Score', 0)}",
                    s.get('Chart', '-'),
                    f"{risk_pct:.1f}%",
                    f"{s['Lot']} lot",
                    f"Rp {s['Cost']:,}"
                ])
                headers = [
                    "Kode", "Sektor", "Harga", "RSI", "Vol", "Flow", "Acc",
                    "Support", "Resist", "SL", "TP", "R/R", "Prob",
                    "EV%", "Skor", "Trend", "Risk%", "Lot", "Biaya"
                ]

        print(tabulate(display_data, headers=headers, tablefmt='grid', stralign='left', numalign='center'))

        total_risk, risk_pct = self.calculate_portfolio_risk(signals)
        print(f"\n📊 Total Portfolio Risk: Rp {total_risk:,} ({risk_pct:.2f}% dari modal)")

        if hasattr(self.config, 'MAX_PORTFOLIO_RISK_PCT') and risk_pct > self.config.MAX_PORTFOLIO_RISK_PCT:
            print(f"⚠️  PERINGATAN: Total risk ({risk_pct:.1f}%) melebihi cap ({self.config.MAX_PORTFOLIO_RISK_PCT}%)!")

    def print_portfolio_guide(self, signals, engine_name):
        if not signals:
            return
        print("\n" + "="*100)
        print(f"📊 {engine_name} - PANDUAN PORTOFOLIO")
        print("="*100)
        print(f"Anda bisa membeli maksimal {getattr(self.config, 'MAX_PORTFOLIO', 5)} saham")
        total_risk_used = 0
        for i, s in enumerate(signals, 1):
            risk_amount = s.get('Risk', 0) * s.get('Lot', 1) * 100
            quality = s.get('Quality', 'N/A')
            ma200_bonus = "⭐" if s.get('MA200_Bonus', 0) else ""
            print(f"  {i}. {s['Symbol']}: {s['Lot']} lot, Harga Rp {s['Price']:,}, Risk Rp {risk_amount:,} {ma200_bonus} {quality}")
            total_risk_used += risk_amount

        print(f"\n💰 Total risk digunakan: Rp {total_risk_used:,}")

    def print_market_regime(self):
        """Print market regime analysis menggunakan IHSG"""
        print("\n" + "="*100)
        print("📊 MARKET REGIME ANALYSIS (IHSG)")
        print("="*100)

        trend, adx = self.market_regime.detect_trend_strength()
        vol_regime, vol_ratio = self.market_regime.detect_volatility_regime()

        print(f"📈 Market Trend: {trend} (ADX: {adx:.1f})")
        print(f"📊 Volatility Regime: {vol_regime} (Ratio: {vol_ratio:.2f})")

# =============================================================================
# 26. PERFORMANCE DASHBOARD
# =============================================================================

class PerformanceDashboard:
    """Dashboard untuk visualisasi performa trading"""

    def __init__(self):
        self.logger = ColabLogger()
        self.dpi = 100
        self.figsize = (12, 8)

    def calculate_metrics(self, trades):
        if not trades:
            return {}

        df_trades = pd.DataFrame(trades)

        total_trades = len(df_trades)
        winning_trades = len(df_trades[df_trades['return_after_fee_pct'] > 0])
        losing_trades = len(df_trades[df_trades['return_after_fee_pct'] <= 0])
        win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

        total_return = df_trades['return_after_fee_pct'].sum()
        avg_return = df_trades['return_after_fee_pct'].mean()

        gross_profit = df_trades[df_trades['return_after_fee_pct'] > 0]['return_after_fee_pct'].sum()
        gross_loss = abs(df_trades[df_trades['return_after_fee_pct'] < 0]['return_after_fee_pct'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

        returns_series = df_trades['return_after_fee_pct'] / 100
        sharpe = returns_series.mean() / returns_series.std() * np.sqrt(252) if returns_series.std() > 0 else 0

        downside_returns = returns_series[returns_series < 0]
        sortino = returns_series.mean() / downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 and downside_returns.std() > 0 else 0

        equity_curve = (1 + returns_series).cumprod()
        running_max = equity_curve.cummax()
        drawdown = (equity_curve - running_max) / running_max * 100
        max_drawdown = drawdown.min()

        return {
            'total_trades': total_trades,
            'winning_trades': winning_trades,
            'losing_trades': losing_trades,
            'win_rate': win_rate,
            'total_return': total_return,
            'avg_return': avg_return,
            'profit_factor': profit_factor,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino,
            'max_drawdown': max_drawdown,
            'gross_profit': gross_profit,
            'gross_loss': gross_loss
        }

    def plot_equity_curve(self, trades, title="Equity Curve", save_path=None):
        if not trades:
            print("No trades to plot")
            return

        df_trades = pd.DataFrame(trades)
        df_trades = df_trades.sort_values('entry_date')

        returns = df_trades['return_after_fee_pct'].values / 100
        equity = 100 * (1 + returns).cumprod()

        plt.figure(figsize=self.figsize, dpi=self.dpi)
        plt.plot(df_trades['entry_date'], equity, linewidth=2, color='blue')
        plt.title(title, fontsize=16, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Equity (Base 100)', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=self.dpi, bbox_inches='tight')

        plt.show()

    def plot_drawdown(self, trades, title="Drawdown Chart", save_path=None):
        if not trades:
            print("No trades to plot")
            return

        df_trades = pd.DataFrame(trades)
        df_trades = df_trades.sort_values('entry_date')

        returns = df_trades['return_after_fee_pct'].values / 100
        equity = (1 + returns).cumprod()
        running_max = np.maximum.accumulate(equity)
        drawdown = (equity - running_max) / running_max * 100

        plt.figure(figsize=self.figsize, dpi=self.dpi)
        plt.fill_between(df_trades['entry_date'], drawdown, 0, color='red', alpha=0.3)
        plt.plot(df_trades['entry_date'], drawdown, color='red', linewidth=1)
        plt.title(title, fontsize=16, fontweight='bold')
        plt.xlabel('Date', fontsize=12)
        plt.ylabel('Drawdown (%)', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=self.dpi, bbox_inches='tight')

        plt.show()

    def plot_monthly_returns(self, trades, title="Monthly Returns Heatmap", save_path=None):
        if not trades:
            print("No trades to plot")
            return

        df_trades = pd.DataFrame(trades)
        df_trades['entry_date'] = pd.to_datetime(df_trades['entry_date'])
        df_trades['year'] = df_trades['entry_date'].dt.year
        df_trades['month'] = df_trades['entry_date'].dt.month

        monthly_returns = df_trades.groupby(['year', 'month'])['return_after_fee_pct'].sum().unstack()

        plt.figure(figsize=self.figsize, dpi=self.dpi)
        sns.heatmap(monthly_returns, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
                    linewidths=0.5, cbar_kws={'label': 'Return (%)'})
        plt.title(title, fontsize=16, fontweight='bold')
        plt.xlabel('Month', fontsize=12)
        plt.ylabel('Year', fontsize=12)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=self.dpi, bbox_inches='tight')

        plt.show()

    def plot_win_loss_distribution(self, trades, title="Win/Loss Distribution", save_path=None):
        if not trades:
            print("No trades to plot")
            return

        df_trades = pd.DataFrame(trades)
        wins = df_trades[df_trades['return_after_fee_pct'] > 0]['return_after_fee_pct']
        losses = df_trades[df_trades['return_after_fee_pct'] <= 0]['return_after_fee_pct']

        plt.figure(figsize=self.figsize, dpi=self.dpi)
        plt.hist(wins, bins=20, alpha=0.7, label=f'Wins (n={len(wins)})', color='green')
        plt.hist(losses, bins=20, alpha=0.7, label=f'Losses (n={len(losses)})', color='red')
        plt.title(title, fontsize=16, fontweight='bold')
        plt.xlabel('Return (%)', fontsize=12)
        plt.ylabel('Frequency', fontsize=12)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=self.dpi, bbox_inches='tight')

        plt.show()

    def print_performance_summary(self, trades):
        metrics = self.calculate_metrics(trades)

        if not metrics:
            print("\n❌ Belum ada data backtest.")
            print("\n📋 CARA MENGGUNAKAN DASHBOARD:")
            print("   1. Jalankan scan saham (Mode 1)")
            print("   2. Pilih engine trading")
            print("   3. Pada opsi backtest, pilih periode (1 atau 2)")
            print("   4. Setelah backtest selesai, kembali ke dashboard")
            print("\n   Atau jika sudah pernah backtest, pastikan state tersimpan.")
            return

        print("\n" + "="*80)
        print("📈 PERFORMANCE SUMMARY")
        print("="*80)

        print(f"\n📊 TRADING STATISTICS:")
        print(f"   Total Trades: {metrics['total_trades']}")
        print(f"   Winning Trades: {metrics['winning_trades']}")
        print(f"   Losing Trades: {metrics['losing_trades']}")
        print(f"   Win Rate: {metrics['win_rate']:.2f}%")

        print(f"\n💰 RETURN STATISTICS:")
        print(f"   Total Return: {metrics['total_return']:.2f}%")
        print(f"   Average Return: {metrics['avg_return']:.2f}%")
        print(f"   Gross Profit: {metrics['gross_profit']:.2f}%")
        print(f"   Gross Loss: {metrics['gross_loss']:.2f}%")
        print(f"   Profit Factor: {metrics['profit_factor']:.2f}")

        print(f"\n📉 RISK STATISTICS:")
        print(f"   Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
        print(f"   Sortino Ratio: {metrics['sortino_ratio']:.2f}")
        print(f"   Max Drawdown: {metrics['max_drawdown']:.2f}%")

        print(f"\n🔍 INTERPRETATION:")
        if metrics['sharpe_ratio'] > 1:
            print(f"   ✅ Sharpe > 1: Good risk-adjusted returns")
        elif metrics['sharpe_ratio'] > 0:
            print(f"   🟡 Sharpe 0-1: Acceptable")
        else:
            print(f"   ❌ Sharpe < 0: Poor risk-adjusted returns")

        if metrics['profit_factor'] > 2:
            print(f"   ✅ Profit Factor > 2: Very profitable")
        elif metrics['profit_factor'] > 1.5:
            print(f"   🟡 Profit Factor 1.5-2: Good")
        elif metrics['profit_factor'] > 1:
            print(f"   🟢 Profit Factor 1-1.5: Profitable")
        else:
            print(f"   ❌ Profit Factor < 1: Not profitable")

        if metrics['max_drawdown'] > -20:
            print(f"   ✅ Max Drawdown > -20%: Acceptable")
        elif metrics['max_drawdown'] > -30:
            print(f"   🟡 Max Drawdown -20% to -30%: High risk")
        else:
            print(f"   ❌ Max Drawdown < -30%: Very high risk")

# =============================================================================
# 27. SENSITIVITY ANALYZER
# =============================================================================

class SensitivityAnalyzer:
    def __init__(self):
        self.results = {}
        self.logger = ColabLogger()

    def analyze_ma200_tolerance(self, df, param_name='ma200_tolerance', values=[0, 0.02, 0.05, 0.08, 0.10, 0.15]):
        results = []
        for tol in values:
            signals = []
            for i in range(200, len(df) - 20, 5):
                data = df.iloc[:i]
                close = data['Close'].iloc[-1]
                ma200 = data['Close'].tail(200).mean()

                if close > ma200 * (1 - tol):
                    signals.append(1)

            win_rate = len(signals) / ((len(df) - 200) / 5) * 100 if signals else 0
            results.append(win_rate)

        stability = np.std(results) / (np.mean(results) + 1e-6)

        self.results[param_name] = {
            'values': values,
            'results': results,
            'stability': stability,
            'is_stable': stability < 0.3
        }

        return self.results[param_name]

    def analyze_ma50_pullback(self, df, param_name='ma50_pullback',
                              lower_values=[-25, -20, -15, -10, -5],
                              upper_values=[5, 10, 15, 20, 25]):
        results_lower = []
        results_upper = []

        for low in lower_values:
            signals = []
            for i in range(100, len(df) - 20, 5):
                data = df.iloc[:i]
                close = data['Close'].iloc[-1]
                ma50 = data['Close'].tail(50).mean()

                pct = (close / ma50 - 1) * 100
                if pct > low and pct < 10:
                    signals.append(1)

            win_rate = len(signals) / ((len(df) - 100) / 5) * 100 if signals else 0
            results_lower.append(win_rate)

        for up in upper_values:
            signals = []
            for i in range(100, len(df) - 20, 5):
                data = df.iloc[:i]
                close = data['Close'].iloc[-1]
                ma50 = data['Close'].tail(50).mean()

                pct = (close / ma50 - 1) * 100
                if pct > -15 and pct < up:
                    signals.append(1)

            win_rate = len(signals) / ((len(df) - 100) / 5) * 100 if signals else 0
            results_upper.append(win_rate)

        stability_lower = np.std(results_lower) / (np.mean(results_lower) + 1e-6)
        stability_upper = np.std(results_upper) / (np.mean(results_upper) + 1e-6)

        self.results[param_name] = {
            'lower_values': lower_values,
            'lower_results': results_lower,
            'upper_values': upper_values,
            'upper_results': results_upper,
            'stability_lower': stability_lower,
            'stability_upper': stability_upper,
            'is_stable': stability_lower < 0.3 and stability_upper < 0.3
        }

        return self.results[param_name]

    def print_summary(self):
        print("\n" + "="*100)
        print("📊 SENSITIVITY ANALYSIS RESULTS")
        print("="*100)

        for param, result in self.results.items():
            print(f"\n🔍 Parameter: {param}")
            if param == 'ma200_tolerance':
                print(f"   Values: {result['values']}")
                formatted_results = [f"{r:.1f}" for r in result['results']]
                print(f"   Results: {formatted_results}")
                print(f"   Stability: {result['stability']:.3f}")
                print(f"   Status: {'✅ STABLE' if result['is_stable'] else '❌ UNSTABLE - OVERFIT RISK!'}")
                if not result['is_stable']:
                    print(f"   ⚠️  RECOMMENDATION: Use strict 0% tolerance")
            elif param == 'ma50_pullback':
                print(f"   Lower bounds: {result['lower_values']}")
                formatted_lower = [f"{r:.1f}" for r in result['lower_results']]
                print(f"   Lower results: {formatted_lower}")
                print(f"   Stability lower: {result['stability_lower']:.3f}")
                print(f"   Upper bounds: {result['upper_values']}")
                formatted_upper = [f"{r:.1f}" for r in result['upper_results']]
                print(f"   Upper results: {formatted_upper}")
                print(f"   Stability upper: {result['stability_upper']:.3f}")
                print(f"   Status: {'✅ STABLE' if result['is_stable'] else '❌ UNSTABLE - OVERFIT RISK!'}")
                if result['is_stable']:
                    print(f"   ✅ RECOMMENDATION: Use range -20% to +15%")

# =============================================================================
# 28. OUT-OF-SAMPLE TESTER
# =============================================================================

class OutOfSampleTester:
    def __init__(self, engine_class, config, global_fetcher):
        self.engine_class = engine_class
        self.config = config
        self.global_fetcher = global_fetcher
        self.logger = ColabLogger()

    def split_data(self, df, train_end_date='2024-01-01'):
        if len(df) < 300:
            return None, None

        train = df[df.index < train_end_date].copy()
        test = df[df.index >= train_end_date].copy()

        if len(test) < 50:
            return self.walk_forward_split(df)

        return train, test

    def walk_forward_split(self, df):
        total_len = len(df)
        train_size = int(total_len * 0.7)
        test_size = total_len - train_size

        if train_size < 200 or test_size < 50:
            return None, None

        train = df.iloc[:train_size]
        test = df.iloc[train_size:]

        self.logger.info(f"   Using walk-forward split: {len(train)} train, {len(test)} test")
        return train, test

    def test_parameter_robustness(self, stocks_data, train_end='2024-01-01'):
        print(f"\n📊 Running Out-of-Sample Test...")
        print(f"   Training: before {train_end} (or 70% walk-forward)")
        print(f"   Testing: after {train_end} (or 30% walk-forward)")

        all_results = []
        total_symbols = 0
        valid_symbols = 0

        for symbol, df in stocks_data.items():
            total_symbols += 1
            if len(df) < 300:
                continue

            train_df, test_df = self.split_data(df, train_end)

            if train_df is None or test_df is None or len(train_df) < 200 or len(test_df) < 50:
                continue

            valid_symbols += 1

            train_signals = []
            for i in range(200, len(train_df), 10):
                data = train_df.iloc[:i]
                engine = self.engine_class(self.config, self.global_fetcher)
                signal = engine.get_signal(symbol, data)
                if signal:
                    train_signals.append(signal)

            train_signal_rate = len(train_signals) / ((len(train_df) - 200) / 10) * 100 if len(train_df) > 200 else 0

            test_signals = []
            for i in range(200, len(test_df), 10):
                data = test_df.iloc[:i]
                engine = self.engine_class(self.config, self.global_fetcher)
                signal = engine.get_signal(symbol, data)
                if signal:
                    test_signals.append(signal)

            test_signal_rate = len(test_signals) / ((len(test_df) - 200) / 10) * 100 if len(test_df) > 200 else 0

            if train_signal_rate > 0 or test_signal_rate > 0:
                all_results.append({
                    'symbol': symbol,
                    'train_rate': train_signal_rate,
                    'test_rate': test_signal_rate,
                    'degradation': (train_signal_rate - test_signal_rate) if train_signal_rate > 0 else 0
                })

        if not all_results:
            print(f"   ❌ No valid data for out-of-sample test (tried {total_symbols} symbols, {valid_symbols} had sufficient data)")
            return None

        df_results = pd.DataFrame(all_results)

        avg_train = df_results['train_rate'].mean()
        avg_test = df_results['test_rate'].mean()
        avg_degradation = df_results['degradation'].mean()

        print(f"\n📈 OUT-OF-SAMPLE TEST RESULTS:")
        print(f"   Symbols tested: {len(all_results)}/{total_symbols}")
        print(f"   Average signal rate - Training: {avg_train:.2f}%")
        print(f"   Average signal rate - Testing: {avg_test:.2f}%")
        print(f"   Average degradation: {avg_degradation:.2f}%")

        if avg_degradation > 20:
            print(f"   ⚠️  High degradation! Potential overfitting!")
        elif avg_degradation > 10:
            print(f"   🟡 Moderate degradation")
        else:
            print(f"   ✅ Low degradation - Strategy robust!")

        return {
            'avg_train': avg_train,
            'avg_test': avg_test,
            'avg_degradation': avg_degradation,
            'details': df_results
        }

# =============================================================================
# 29. MAIN PROGRAM - V3.4 (TANPA TEST MODE) - FIXED
# =============================================================================

def main():
    print("\n" + "="*60)
    print("🏦 IDX STOCK SCANNER - V3.4 (PHASE 3 - FINAL)")
    print("   ADX Fix | Filter Modal Kecil | Google Sheets URL Tetap")
    print("="*60)

    logger = ColabLogger()
    memory_manager = MemoryManager(threshold_mb=1000)
    keep_alive = KeepAlive(interval=120)

    config_manager = ConfigManager()
    sheets_exporter = GoogleSheetsExporter()

    ka_choice = input("\nAktifkan keep-alive? (y/n, default y): ").strip().lower()
    if ka_choice != 'n':
        keep_alive.start()

    # Tampilkan URL Google Sheets di awal (meski 0 sinyal)
    sheets_exporter.ensure_spreadsheet_exists()

    print("\nPilih mode:")
    print("1. Scan Saham (Normal)")
    print("2. Sensitivity Analysis (Cek Overfitting)")
    print("3. Out-of-Sample Test (Validasi Strategi)")
    print("4. Performance Dashboard (Lihat Performa)")
    print("5. Config Management (Atur Parameter)")

    mode_choice = input("Pilihan (1/2/3/4/5): ").strip()

    # ========== CONFIG MANAGEMENT ==========
    if mode_choice == '5':
        print("\n⚙️ CONFIGURATION MANAGEMENT")
        print("="*60)

        config_manager.print_config()

        print("\nPilihan:")
        print("1. Edit Engine Config")
        print("2. Reset to Default")
        print("3. List Backups")
        print("4. Restore from Backup")
        print("0. Back to Main")

        cfg_choice = input("Pilihan: ").strip()

        if cfg_choice == '1':
            print("\nPilih engine:")
            print("1. Swing")
            print("2. Intraday")
            print("3. Gorengan")
            print("4. Investasi")
            engine_choice = input("Pilihan (1-4): ").strip()

            engine_map = {'1': 'swing', '2': 'intraday', '3': 'gorengan', '4': 'investasi'}
            if engine_choice in engine_map:
                engine_name = engine_map[engine_choice]
                print(f"\nCurrent {engine_name} config:")
                print(json.dumps(config_manager.get_engine_config(engine_name), indent=2))

                print("\nMasukkan parameter baru (format JSON):")
                print("Contoh: {\"min_price\": 50, \"max_spread_pct\": 7.0}")
                try:
                    updates = json.loads(input("Updates: "))
                    config_manager.update_engine_config(engine_name, updates)
                except:
                    print("❌ Invalid JSON")

        elif cfg_choice == '2':
            print("\nReset to default:")
            print("1. All engines")
            print("2. Specific engine")
            reset_choice = input("Pilihan: ").strip()

            if reset_choice == '1':
                config_manager.reset_to_default()
            elif reset_choice == '2':
                engine_name = input("Engine name (swing/intraday/gorengan/investasi): ").strip()
                config_manager.reset_to_default(engine_name)

        elif cfg_choice == '3':
            backups = config_manager.list_backups()
            if backups:
                print("\n📋 Available Backups:")
                for b in backups:
                    print(f"   {b['file']} - {b['time']} ({b['size']} bytes)")
            else:
                print("No backups found")

        elif cfg_choice == '4':
            backups = config_manager.list_backups()
            if backups:
                print("\nPilih backup file:")
                for i, b in enumerate(backups, 1):
                    print(f"{i}. {b['file']}")

                try:
                    idx = int(input("Pilihan: ")) - 1
                    if 0 <= idx < len(backups):
                        config_manager.restore_backup(backups[idx]['file'])
                except:
                    print("❌ Invalid choice")

        keep_alive.stop()
        return

    # ========== PERFORMANCE DASHBOARD ==========
    elif mode_choice == '4':
        print("\n📈 PERFORMANCE DASHBOARD")
        print("="*60)

        state_manager = ColabStateManager()
        states = state_manager.load_history(days=30)

        all_trades = []
        for state in states:
            trades = state.get('data', {}).get('trades', [])
            all_trades.extend(trades)

        dashboard = PerformanceDashboard()
        dashboard.print_performance_summary(all_trades)

        if all_trades:
            print("\n📊 Generating charts...")
            dashboard.plot_equity_curve(all_trades, title="Equity Curve (Last 30 Days)")
            dashboard.plot_drawdown(all_trades, title="Drawdown Chart")

            if len(all_trades) >= 20:
                dashboard.plot_win_loss_distribution(all_trades)
                dashboard.plot_monthly_returns(all_trades)

            save_choice = input("\n💾 Save charts to Drive? (y/n): ").strip().lower()
            if save_choice == 'y':
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                dashboard.plot_equity_curve(all_trades, save_path=f"/content/drive/MyDrive/equity_curve_{timestamp}.png")
                dashboard.plot_drawdown(all_trades, save_path=f"/content/drive/MyDrive/drawdown_{timestamp}.png")
                print(f"✅ Charts saved to Google Drive")

        keep_alive.stop()
        return

    # ========== SENSITIVITY ANALYSIS ==========
    elif mode_choice == '2':
        print("\n🔬 Running Sensitivity Analysis...")
        config = InvestasiConfig(100000)
        global_fetcher = GlobalIndicesFetcher()
        global_fetcher.fetch_all(config)

        fetcher = OptimizedDataFetcher(max_workers=10)
        sample_symbols = ['BBCA', 'BBRI', 'BMRI', 'ASII', 'TLKM']
        stocks_data = fetcher.fetch_parallel(sample_symbols, config)

        if stocks_data:
            analyzer = SensitivityAnalyzer()

            for symbol, df in stocks_data.items():
                if len(df) > 300:
                    print(f"\n📈 Analyzing {symbol}...")
                    analyzer.analyze_ma200_tolerance(df)
                    analyzer.analyze_ma50_pullback(df)

            analyzer.print_summary()

            print("\n" + "="*100)
            print("🎯 FINAL RECOMMENDATION (Based on Sensitivity Analysis)")
            print("="*100)
            print("✅ MA200 Tolerance: USE 0% (STRICT) - parameter unstable")
            print("✅ MA50 Pullback: USE -20% to +15% - parameter stable")
            print("\n⚠️  Never optimize parameters based on backtest results!")

        keep_alive.stop()
        return

    # ========== OUT-OF-SAMPLE TEST ==========
    elif mode_choice == '3':
        print("\n🔬 Running Out-of-Sample Test...")
        config = InvestasiConfig(100000)
        global_fetcher = GlobalIndicesFetcher()
        global_fetcher.fetch_all(config)

        fetcher = OptimizedDataFetcher(max_workers=10)
        universe = QUALITY_STOCKS[:50]
        stocks_data = fetcher.fetch_parallel(universe, config)

        if stocks_data:
            tester = OutOfSampleTester(InvestasiEngine, config, global_fetcher)
            results = tester.test_parameter_robustness(stocks_data, train_end='2024-01-01')

            if results and results['avg_degradation'] > 20:
                print("\n⚠️  WARNING: High degradation detected!")
            elif results and results['avg_degradation'] < 10:
                print("\n✅ Strategy appears robust!")

        keep_alive.stop()
        return

    # ========== NORMAL SCAN ==========
    elif mode_choice == '1':
        print("\nPilih engine trading:")
        print("1. Swing Engine (Mingguan - Mean Reversion)")
        print("2. Intraday Liquid (1 jam - Momentum)")
        print("3. Intraday Gorengan (1 jam - Early Momentum)")
        print("4. Investasi Engine (Quality + Trend - Jangka Panjang)")

        while True:
            engine_choice = input("Pilihan (1/2/3/4): ").strip()
            if engine_choice in ['1', '2', '3', '4']:
                break
            print("❌ Pilih 1, 2, 3, atau 4")

        while True:
            try:
                modal_input = input("\nModal (Rp 10,000 - 5,000,000): ").strip()
                modal = int(modal_input.replace('.', '').replace(',', ''))
                if 10000 <= modal <= 5000000:
                    break
                print("❌ Modal harus 10,000 - 5,000,000")
            except:
                print("❌ Input tidak valid")

        # Load config dari ConfigManager
        engine_map = {'1': 'swing', '2': 'intraday', '3': 'gorengan', '4': 'investasi'}
        engine_name_config = engine_map[engine_choice]
        engine_config = config_manager.get_engine_config(engine_name_config)

        # Buat config berdasarkan engine
        if engine_choice == '1':
            mode = 'mingguan'
            config = Config(modal, mode)
            for key, value in engine_config.items():
                if hasattr(config, key):
                    setattr(config, key, value)
            engine_name = "SWING ENGINE"
            EngineClass = SwingEngine
        elif engine_choice == '2':
            mode = 'intraday'
            config = Config(modal, mode)
            for key, value in engine_config.items():
                if hasattr(config, key):
                    setattr(config, key, value)
            engine_name = "INTRADAY LIQUID ENGINE"
            EngineClass = IntradayLiquidEngine
        elif engine_choice == '3':
            mode = 'intraday'
            config = Config(modal, mode)
            # Override untuk gorengan (tetap ada karena ini default)
            config.MIN_PRICE = 50
            config.MAX_SPREAD_PCT = 3.0
            config.MIN_AVG_VOLUME = 100000
            config.MIN_EV_PCT = 1.5
            config.RISK_PER_TRADE_PCT = 0.3
            config.MAX_TRADES_PER_DAY = 2
            # Override dengan config dari file
            for key, value in engine_config.items():
                if hasattr(config, key):
                    setattr(config, key, value)
            engine_name = "INTRADAY GORENGAN ENGINE"
            EngineClass = IntradayGorenganEngine
        else:
            config = InvestasiConfig(modal)
            for key, value in engine_config.items():
                if hasattr(config, key):
                    setattr(config, key, value)
            engine_name = "INVESTASI ENGINE"
            EngineClass = InvestasiEngine

        if hasattr(config, 'ENABLE_RANDOM_SLIPPAGE') and config.ENABLE_RANDOM_SLIPPAGE:
            FeeConfig.SLIPPAGE_MODE = 'random'
        else:
            FeeConfig.SLIPPAGE_MODE = 'fixed'

        state_manager = ColabStateManager()
        global_fetcher = GlobalIndicesFetcher()
        global_fetcher.fetch_all(config)

        engine = EngineClass(config, global_fetcher)
        fetcher = OptimizedDataFetcher(max_workers=10)

        if engine_choice == '4':
            universe = config.QUALITY_STOCKS
            print(f"\n📊 Quality Universe: {len(universe)} saham")
        else:
            universe = STOCKBIT_UNIVERSE
            print(f"\n📊 Universe: {len(universe)} saham")

        print(f"\n💰 Modal: Rp {modal:,}")
        print(f"📊 Engine: {engine_name}")
        print(f"📊 Interval: {getattr(config, 'INTERVAL', '1d')}")
        if hasattr(config, 'MAX_PORTFOLIO'):
            print(f"📊 Max posisi: {config.MAX_PORTFOLIO}")
        if hasattr(config, 'MIN_PRICE'):
            print(f"📊 Min Price: Rp {config.MIN_PRICE}")
        if hasattr(config, 'MAX_SPREAD_PCT'):
            print(f"📊 Max Spread: {config.MAX_SPREAD_PCT}%")
        if hasattr(config, 'MIN_AVG_VOLUME'):
            print(f"📊 Min Avg Volume: {config.MIN_AVG_VOLUME:,}")

        print(f"\n📥 Fetching {len(universe)} stocks with {fetcher.max_workers} workers...")
        start_time = time.time()

        stocks_data = fetcher.fetch_parallel(universe, config)

        elapsed = time.time() - start_time
        fetcher.print_stats()

        if stocks_data:
            print(f"\n📊 Menganalisis {len(stocks_data)} saham...")
            signals = []

            for symbol, df in stocks_data.items():
                signal = engine.get_signal(symbol, df)
                if signal:
                    signals.append(signal)

                if len(signals) % 50 == 0 and len(signals) > 0:
                    memory_manager.log_memory(f"after {len(signals)} signals")

            print(f"✅ Ditemukan {len(signals)} sinyal mentah")

            scanner = StockScanner(config, global_fetcher, engine, state_manager)

            # Market regime analysis (fixed)
            scanner.print_market_regime()

            top_sectors, sector_perf = scanner.market_regime.detect_sector_rotation(stocks_data)
            if top_sectors:
                print(f"\n🔥 Top 3 Sectors (20d return):")
                for sector, ret in top_sectors:
                    print(f"   {sector}: {ret:.2f}%")

            ranked_signals = scanner.filter_by_ranking(signals)
            print(f"✅ Setelah ranking: {len(ranked_signals)} sinyal terbaik")

            scanner.print_global_summary()

            comparison = scanner.compare_with_previous(ranked_signals)
            if comparison:
                scanner.print_comparison(comparison)

            scanner.print_signals(ranked_signals, engine_name)
            scanner.print_portfolio_guide(ranked_signals, engine_name)

            # Google Sheets export untuk semua engine
            if ranked_signals:
                sheets_choice = input(f"\n📊 Export {engine_name} signals ke Google Sheets? (y/n): ").strip().lower()
                if sheets_choice == 'y':
                    sheets_exporter.export_signals(ranked_signals, engine_name, modal, auto_confirm=True)

            print("\n💾 Menyimpan state...")
            state_manager.save_state({
                'signals': ranked_signals,
                'engine': engine_name,
                'modal': modal,
                'trades': [],
                'timestamp': datetime.now(),
                'stats': {
                    'total_stocks': len(stocks_data),
                    'total_signals': len(signals),
                    'fetcher_stats': fetcher.stats
                }
            })

            if engine_choice != '4':
                print("\n" + "="*100)
                print("📊 BACKTEST OPTION")
                print("="*100)
                print("Pilih periode backtest:")
                print("1. 6 bulan (cepat, 10-15 menit)")
                print("2. 5 tahun (lengkap, 2-3 jam)")
                print("0. Lewati")

                bt_choice = input("Pilihan (0/1/2): ").strip()

                if bt_choice in ['1', '2']:
                    time_step = 1
                    if bt_choice == '2' and engine_choice == '1':
                        time_step = 3

                    print("\n" + "="*100)
                    print(f"📊 BACKTEST")
                    print("="*100)  # FIX: Hapus tanda kutip berlebih

                    backtester = Backtester(config, global_fetcher, engine)
                    backtester.run(stocks_data, time_step=time_step)
                    backtester.print_results()

                    if len(backtester.trades) >= 20:
                        print("\n" + "="*100)
                        print("🎲 MONTE CARLO SIMULATION")
                        print("="*100)
                        backtester.run_monte_carlo()
                        backtester.monte_carlo.print_results()

                        state_manager.save_state({
                            'signals': ranked_signals,
                            'trades': backtester.trades,
                            'engine': engine_name,
                            'modal': modal,
                            'timestamp': datetime.now()
                        })
                    else:
                        print(f"\n⚠️ Monte Carlo: Minimal 20 trades required (current: {len(backtester.trades)})")
                else:
                    print("\n✅ Backtest dilewati.")
            else:
                print("\nℹ️  Backtest tidak tersedia untuk Engine Investasi.")

            memory_manager.clear_memory(force=True)

        else:
            print(f"\n❌ Tidak ada data")

        keep_alive.stop()

        print("\n" + "="*100)
        print("📝 CARA EKSEKUSI:")
        print("="*100)
        print("1. Pilih saham yang ingin dibeli (prioritas yang ada ⭐ bonus MA200)")
        print("2. Gunakan LIMIT ORDER di harga ≤ rekomendasi")
        print("3. Pasang Stop Loss sesuai level")
        if engine_choice == '4':
            print("4. Target HOLD jangka panjang (exit jika turun >20%)")
            print("   Info tambahan: PER/PBV hanya untuk referensi")
        else:
            print("4. Target Take Profit sesuai level")
        print("5. Patuhi risk management")
        if engine_choice in ['2', '3']:
            print("6. WAJIB CLOSE sebelum jam 15:50 (no overnight)")
            print(f"7. Max {getattr(config, 'MAX_TRADES_PER_DAY', 2)} trade per hari")
            print("8. Jika loss 2x berturut-turut, stop trading hari itu")
        if engine_choice == '3':
            print("⚠️  PERINGATAN: Saham gorengan berisiko tinggi!")
        if engine_choice == '4':
            print("ℹ️  Investasi jangka panjang: rebalancing bulanan cukup.")
        print("\n✅ Data source: YFinance (optimized with adaptive timeout)")
        print("✅ Cache: 6 jam untuk mempercepat scan berikutnya")
        print("✅ Parameter final: MA200 Strict (0%), MA50 Pullback -20% to +15%")
        print("✅ Filter adaptif berdasarkan modal")
        print("✅ ADX Fixed - menggunakan IHSG")
        print("✅ Google Sheets URL: Sudah tampil di atas")
        print("✅ Selamat trading!")

    else:
        print("❌ Pilihan tidak valid")
        keep_alive.stop()

if __name__ == "__main__":
    main()

🏦 IDX STOCK SCANNER - V3.4 (PHASE 3 - FINAL FIX)
   ADX Fix | Filter Modal Kecil | Google Sheets URL Tetap

🏦 IDX STOCK SCANNER - V3.4 (PHASE 3 - FINAL)
   ADX Fix | Filter Modal Kecil | Google Sheets URL Tetap
ℹ️ Config loaded from config/scanner_config.json

Aktifkan keep-alive? (y/n, default y): y
⏱️ Keep-alive: 23:11:20
🟢 Keep-alive started (interval 120s)
ℹ️ Opened existing spreadsheet: IDX_Scanner_All_Engines

📊 Google Sheets URL: https://docs.google.com/spreadsheets/d/10ITzuSx_6512KrfSmkqaRPaAsFbPi6Z_NO-6e6u9BKg

Pilih mode:
1. Scan Saham (Normal)
2. Sensitivity Analysis (Cek Overfitting)
3. Out-of-Sample Test (Validasi Strategi)
4. Performance Dashboard (Lihat Performa)
5. Config Management (Atur Parameter)
Pilihan (1/2/3/4/5): 1

Pilih engine trading:
1. Swing Engine (Mingguan - Mean Reversion)
2. Intraday Liquid (1 jam - Momentum)
3. Intraday Gorengan (1 jam - Early Momentum)
4. Investasi Engine (Quality + Trend - Jangka Panjang)
Pilihan (1/2/3/4): 3

Modal (Rp 10,000 - 5,000